# TinyGestureSSD — camera-ready revision (steps 0 and 1)

Run from the **project root** (the folder that contains `dataset/`, `models/`, `models_kd/`, `models_width_sweep/`, `models_pretrained/`), same as the original notebooks.

* **Step 0** — definition cells copied *verbatim* from `TinyGestureSSD_kd.ipynb` (cell index noted in each header). Nothing retyped, so every new number comes from the same metric code as the paper.
* **Step 1** — fp32 re-evaluation of every saved checkpoint on the existing 433-image validation split. **Inference only, no training.**

Outputs go to `revision_results/`. `SEED` stays 0 so the cached split is identical to the paper's; later seed runs must only change init/batch order, never the split.

## Step 0 — verbatim definitions

*verbatim: KD notebook cell [4] — config + set_seed + device*

*verbatim: KD notebook cell [6] — anchors + box utils*

*verbatim: KD notebook cell [7] — anchor tensors*

*verbatim: KD notebook cell [10] — backbone*

*verbatim: KD notebook cell [12] — head + detector*

*verbatim: KD notebook cell [15] — box encode/decode/matching*

*verbatim: KD notebook cell [18] — MultiBoxLoss*

*verbatim: KD notebook cell [21] — dataset + prepare_dataset*

*verbatim: KD notebook cell [22] — build loaders*

*verbatim: KD notebook cell [24] — refit anchors*

*verbatim: KD notebook cell [28] — predict_batch_labels / evaluate / train_model*

### Revision setup: output dir + split fingerprint

## Step 1 — fp32 re-evaluation (inference only)

Checkpoint map is explicit because file names are inconsistent (`model_*` = scratch sweep, `student_*` = KD sweep).
**Scratch width 0.3 has no `.pt` in the upload**, so it is evaluated from `models/gesture_ssdlite.onnx` (BatchNorm already folded, same fp32 function). If you find the original `.pt`, drop its path into `SCRATCH_030_PT` and it will be used instead, with the ONNX kept as a cross-check.

In [38]:
import onnxruntime as ort
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SCRATCH_030_PT = None   # e.g. "models/best_model_w0p3.pt" if you recover it
SW = "models_width_sweep"
CKPTS = {   # key: (kind, width, path)
    "scratch_0.25": ("scratch", 0.25, f"{SW}/model_w0p25_best.pt"),
    "scratch_0.50": ("scratch", 0.50, f"{SW}/model_w0p5_best.pt"),
    "scratch_0.75": ("scratch", 0.75, f"{SW}/model_w0p75_best.pt"),
    "scratch_1.00": ("scratch", 1.00, f"{SW}/model_w1p0_best.pt"),
    "kd_0.25":      ("kd", 0.25, f"{SW}/student_w0p25_best.pt"),
    "kd_0.30":      ("kd", 0.30, "models_kd/best_model.pt"),
    "kd_0.50":      ("kd", 0.50, f"{SW}/student_w0p5_best.pt"),
    "kd_0.750.y
     0.":      ("kd", 0.75, f"{SW}/student_w0p75_best.pt"), 
    "pretrained_1.00": ("pretrained", 1.00, "models_pretrained/best_model.pt"),
    "teacher_1.00":    ("teacher", 1.00, "models_kd/teacher_best.pt"),
}
if SCRATCH_030_PT: CKPTS["scratch_0.30"] = ("scratch", 0.30, SCRATCH_030_PT)

def metrics_from_preds(preds, gts):
    """Same formulas as evaluate(); a miss is encoded as NUM_CLASSES here (counts as wrong)."""
    acc = accuracy_score(gts, preds)
    p, r, f1, _ = precision_recall_fscore_support(gts, preds, labels=list(range(NUM_CLASSES)),
                                                  average="macro", zero_division=0)
    return {"accuracy": float(acc), "precision": float(p), "recall": float(r), "f1_macro": float(f1),
            "n_missed": int((preds == NUM_CLASSES).sum()), "n_total": int(len(gts))}

@torch.no_grad()
def torch_preds(model):
    model.eval(); out = []
    for imgs, _, _ in val_loader:
        p = predict_batch_labels(model, imgs.to(device), anchors_center_d).cpu()
        out.append(torch.where(p == -1, torch.full_like(p, NUM_CLASSES), p))
    return torch.cat(out).numpy()

def onnx_preds(path, conf_thresh=0.5):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"]); out = []
    for img, _, _ in val_loader.dataset:                       # ONNX is batch=1
        cls, _loc = sess.run(None, {"input": img.unsqueeze(0).numpy()})
        probs = torch.softmax(torch.from_numpy(cls), dim=-1)[..., 1:]
        score, c = probs.max(dim=-1)                           # (1, A)
        score = score.masked_fill(score < conf_thresh, -1.0)
        top, idx = score.max(dim=-1)
        out.append(int(c.gather(1, idx.unsqueeze(1)).item()) if top.item() >= 0 else NUM_CLASSES)
    return np.array(out)

results, gts = {}, val_labels
for key, (kind, w, path) in CKPTS.items():
    m = SSDLiteGestureDetector(width_mult=w).to(device)          # (w=1.0 may fetch ImageNet weights, then overwritten)
    m.load_state_dict(torch.load(path, map_location=device))
    preds = torch_preds(m)
    met = metrics_from_preds(preds, gts)
    ref = evaluate(m, val_loader, anchors_center_d, device)      # original metric code, as a cross-check
    assert abs(ref["f1_macro"] - met["f1_macro"]) < 1e-9 and abs(ref["accuracy"] - met["accuracy"]) < 1e-9, key
    results[key] = {"kind": kind, "width": w, "params": sum(p.numel() for p in m.parameters()),
                    "source": path, **met}
    np.save(os.path.join(REV_DIR, "preds", f"fp32_{key}.npy"), preds)
    print(f"{key:16s} params={results[key]['params']:>8,}  F1={met['f1_macro']*100:6.2f}  acc={met['accuracy']*100:6.2f}  missed={met['n_missed']}")

# scratch 0.3 via ONNX (BN folded) -- also the cross-check if a .pt was supplied
p_onnx = onnx_preds("models/gesture_ssdlite.onnx")
met = metrics_from_preds(p_onnx, gts)
key = "scratch_0.30_onnx" if SCRATCH_030_PT else "scratch_0.30"
results[key] = {"kind": "scratch", "width": 0.30, "params": 86870, "source": "models/gesture_ssdlite.onnx (BN folded)", **met}
np.save(os.path.join(REV_DIR, "preds", f"fp32_{key}.npy"), p_onnx)
print(f"{key:16s} params=  86,870  F1={met['f1_macro']*100:6.2f}  acc={met['accuracy']*100:6.2f}  missed={met['n_missed']}")

json.dump(results, open(os.path.join(REV_DIR, "fp32_sweep.json"), "w"), indent=2)
print("\nsaved ->", os.path.join(REV_DIR, "fp32_sweep.json"))

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 27)

### Compare against the numbers currently in the paper

In [14]:
PAPER_F1 = {"scratch_0.25": 19.04, "scratch_0.30": 25.32, "scratch_0.50": 25.87, "scratch_0.75": 36.56,
            "scratch_1.00": 44.31, "teacher_1.00": 86.92, "kd_0.30": 69.78}   # F1 % in paper tex
PAPER_ACC = {"scratch_0.30": 18.48, "teacher_1.00": 79.68, "kd_0.30": 57.51}
print(f"{'model':18s}{'F1 now':>9}{'F1 paper':>10}{'dF1':>8}{'acc now':>9}{'acc paper':>10}")
for k, r in results.items():
    f, a = r["f1_macro"]*100, r["accuracy"]*100
    pf, pa = PAPER_F1.get(k), PAPER_ACC.get(k)
    flag = "" if pf is None or abs(f-pf) < 0.011 else "   <-- differs"
    print(f"{k:18s}{f:9.2f}{(pf if pf is not None else float('nan')):10.2f}{(f-pf if pf is not None else float('nan')):8.2f}"
          f"{a:9.2f}{(pa if pa is not None else float('nan')):10.2f}{flag}")
print("\nLooking for: (1) scratch_0.30 -> 25.24 vs 25.32?  (2) pretrained_1.00 vs teacher_1.00 identical?  (3) the 6 red fp32 accuracy cells.")

model                F1 now  F1 paper     dF1  acc now acc paper
scratch_0.25          19.04     19.04   -0.00    12.70       nan
scratch_0.50          25.87     25.87   -0.00    17.55       nan
scratch_0.75          36.48     36.56   -0.08    27.94       nan   <-- differs
scratch_1.00          44.58     44.31    0.27    33.26       nan   <-- differs
kd_0.25               57.43       nan     nan    42.96       nan
kd_0.30               69.78     69.78    0.00    57.51     57.51
kd_0.50               73.00       nan     nan    62.12       nan
kd_0.75               81.33       nan     nan    72.98       nan
pretrained_1.00       86.92       nan     nan    79.68       nan
teacher_1.00          86.92     86.92    0.00    79.68     79.68
scratch_0.30          25.32     25.32   -0.00    18.48     18.48

Looking for: (1) scratch_0.30 -> 25.24 vs 25.32?  (2) pretrained_1.00 vs teacher_1.00 identical?  (3) the 6 red fp32 accuracy cells.


In [16]:
from esp_ppq.api import espdl_quantize_onnx
from esp_ppq.executor import TorchExecutor
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

ESPDL_TARGET = "esp32s3"
ESPDL_NUM_BITS = 8
ESPDL_CALIB_STEPS = min(200, len(val_loader.dataset))
ESPDL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def collate_fn(batch):
    imgs, _boxes, _labels = batch
    return imgs.to(ESPDL_DEVICE)

@torch.no_grad()
def evaluate_espdl(executor, dataset, anchors_center, device, conf_thresh: float = 0.5):
    all_preds, all_gts = [], []
    for img, box, label in dataset:
        img_in = img.unsqueeze(0).to(device)
        outputs = executor(img_in)
        cls_out, loc_out = outputs[0], outputs[1]

        probs = F.softmax(cls_out.float(), dim=-1)
        fg_probs = probs[..., 1:]
        best_score, best_cls = fg_probs.max(dim=-1)
        thresholded = best_score.masked_fill(best_score < conf_thresh, -1.0)
        top_score, top_anchor = thresholded.max(dim=-1)
        pred_cls = best_cls.gather(1, top_anchor.unsqueeze(1)).squeeze(1)
        pred_label = int(pred_cls.item()) if top_score.item() >= 0 else NUM_CLASSES

        all_preds.append(pred_label)
        all_gts.append(int(label))

    all_preds = np.array(all_preds)
    all_gts = np.array(all_gts)
    accuracy = accuracy_score(all_gts, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_gts, all_preds, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)
    n_missed = int((all_preds == NUM_CLASSES).sum())
    return ({"accuracy": accuracy, "precision": precision, "recall": recall,
             "f1_macro": f1, "n_missed": n_missed, "n_total": len(all_gts)},
            all_preds, all_gts)

In [19]:
PTQ_DIR = os.path.join(REV_DIR, "ptq_no_test"); os.makedirs(PTQ_DIR, exist_ok=True)

PTQ_JOBS = {   # key: (onnx, old espdl, old QAT espdl, paper PTQ F1 %, paper PTQ acc %)
    "scratch_0.30":    ("models/gesture_ssdlite.onnx",            "models/gesture_ssdlite.espdl",            "models/gesture_ssdlite_qat.espdl",            20.13, 13.86),
    "kd_0.30":         ("models_kd/gesture_ssdlite.onnx",         "models_kd/gesture_ssdlite.espdl",         "models_kd/gesture_ssdlite_qat.espdl",         67.88, 56.12),
    "pretrained_1.00": ("models_pretrained/gesture_ssdlite.onnx", "models_pretrained/gesture_ssdlite.espdl", "models_pretrained/gesture_ssdlite_qat.espdl", 80.80, 70.44),
}
kb = lambda p: os.path.getsize(p) / 1024

ptq_res = {}
for key, (onnx_p, old_p, qat_p, p_f1, p_acc) in PTQ_JOBS.items():
    out_p = os.path.join(PTQ_DIR, f"{key}.espdl")
    calib = DataLoader(val_loader.dataset, batch_size=1, shuffle=False)       # same calibration data/order as the original export
    graph = espdl_quantize_onnx(
        onnx_import_file=onnx_p, espdl_export_file=out_p,
        calib_dataloader=calib, calib_steps=ESPDL_CALIB_STEPS,
        input_shape=[1, 3, IMG_SIZE, IMG_SIZE], inputs=None, target=ESPDL_TARGET,
        num_of_bits=ESPDL_NUM_BITS, collate_fn=collate_fn, device=ESPDL_DEVICE,
        error_report=False, skip_export=False, export_test_values=False, verbose=0)
    met, preds, gts_ = evaluate_espdl(TorchExecutor(graph=graph, device=ESPDL_DEVICE), val_loader.dataset, anchors_center_d, ESPDL_DEVICE)
    assert (gts_ == val_labels).all()
    np.save(os.path.join(REV_DIR, "preds", f"ptq_{key}.npy"), preds)
    ptq_res[key] = {"onnx": onnx_p, "size_kb_new": kb(out_p), "size_kb_old": kb(old_p),
                    "size_kb_qat_existing": kb(qat_p),
                    "f1_new": met["f1_macro"]*100, "acc_new": met["accuracy"]*100, "n_missed_new": met["n_missed"],
                    "f1_paper": p_f1, "acc_paper": p_acc}
    r = ptq_res[key]
    print(f"{key:16s} size {r['size_kb_old']:7.2f} -> {r['size_kb_new']:7.2f} KB (QAT file {r['size_kb_qat_existing']:7.2f})"
          f" | F1 {r['f1_new']:6.2f} (paper {p_f1:5.2f})  acc {r['acc_new']:6.2f} (paper {p_acc:5.2f})  missed {r['n_missed_new']}")

json.dump(ptq_res, open(os.path.join(REV_DIR, "ptq_reexport.json"), "w"), indent=2)
print("\nsaved ->", os.path.join(REV_DIR, "ptq_reexport.json"))

[15:23:13] ConvTranspose Decomposition Pass Running ... Finished.
[15:23:16] PPQ Quantization Fusion Pass Running ...       Finished.
[15:23:17] PPQ Quantize Simplify Pass Running ...         Finished.
[15:23:17] PPQ Parameter Quantization Pass Running ...    Finished.
[15:23:18] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 200/200 [00:26<00:00,  7.47it/s]


Finished.
[15:24:04] PPQ Quantization Alignment Pass Running ...    Finished.
[15:24:04] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [85]
Num of Quantized Op:          [85]
Num of Variable:              [232]
Num of Quantized Var:         [232]
------- Quantization Snapshot ------
Num of Quant Config:          [324]
ACTIVATED:                    [87]
OVERLAPPED:                   [110]
PASSIVE:                      [123]
FP32:                         [4]
Network Quantization Finished.
[INFO][ESPDL][2026-10-05 15:24:05]:  Skip /m/Constant_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:24:05]:  Skip /m/Constant_2_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:24:05]:  Skip /m/Constant_1_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:24:05]:  Skip /m/Constant_3_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:24:05]:  Skip /m/Constant_output

Calibration Progress(Phase 2): 100%|██████████| 200/200 [00:15<00:00, 13.28it/s]


Finished.
[15:26:11] PPQ Quantization Alignment Pass Running ...    Finished.
[15:26:11] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [85]
Num of Quantized Op:          [85]
Num of Variable:              [232]
Num of Quantized Var:         [232]
------- Quantization Snapshot ------
Num of Quant Config:          [324]
ACTIVATED:                    [87]
OVERLAPPED:                   [110]
PASSIVE:                      [123]
FP32:                         [4]
Network Quantization Finished.
[INFO][ESPDL][2026-10-05 15:26:11]:  Skip /m/Constant_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:26:11]:  Skip /m/Constant_2_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:26:11]:  Skip /m/Constant_1_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:26:11]:  Skip /m/Constant_3_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:26:11]:  Skip /m/Constant_output

Calibration Progress(Phase 2): 100%|██████████| 200/200 [00:15<00:00, 12.87it/s]


Finished.
[15:28:01] PPQ Quantization Alignment Pass Running ...    Finished.
[15:28:01] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [85]
Num of Quantized Op:          [85]
Num of Variable:              [232]
Num of Quantized Var:         [232]
------- Quantization Snapshot ------
Num of Quant Config:          [324]
ACTIVATED:                    [87]
OVERLAPPED:                   [110]
PASSIVE:                      [123]
FP32:                         [4]
Network Quantization Finished.
[INFO][ESPDL][2026-10-05 15:28:02]:  Skip /m/Constant_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:28:02]:  Skip /m/Constant_2_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:28:02]:  Skip /m/Constant_1_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:28:02]:  Skip /m/Constant_3_output_0 because it's not exportable
[INFO][ESPDL][2026-10-05 15:28:02]:  Skip /m/Constant_output

In [20]:
ok_all = True
for key, r in ptq_res.items():
    dF, dA = r["f1_new"] - r["f1_paper"], r["acc_new"] - r["acc_paper"]
    same = abs(dF) < 0.011 and abs(dA) < 0.011
    ok_all &= same
    print(f"{key:16s} dF1={dF:+.2f}  dAcc={dA:+.2f}  dSize={r['size_kb_new']-r['size_kb_old']:+.2f} KB   {'MATCH' if same else '<-- DIFFERS (<=1 image => threshold flip; paste this output)'}")
print("\nALL MATCH: sizes can replace the old PTQ sizes in Tables IV-VI, Fig. 1 and Fig. 2." if ok_all else
      "\nSome rows differ: paste this output before changing any table.")

scratch_0.30     dF1=-0.00  dAcc=-0.00  dSize=-49.70 KB   MATCH
kd_0.30          dF1=+0.00  dAcc=+0.00  dSize=-49.66 KB   MATCH
pretrained_1.00  dF1=-0.58  dAcc=-0.69  dSize=-49.67 KB   <-- DIFFERS (<=1 image => threshold flip; paste this output)

Some rows differ: paste this output before changing any table.


In [21]:
import json, os, glob
import numpy as np

REV = REV_DIR            # from Step 0
ROOT = "."               # project root (models/, models_kd/, models_pretrained/)
B, SEED, NC = 10_000, 12345, 6
P = os.path.join(REV, "preds")
gts = np.load(os.path.join(P, "gts.npy")).astype(int); n = len(gts)

models = {}
for f in sorted(glob.glob(os.path.join(P, "fp32_*.npy"))):
    models["fp32_" + os.path.basename(f)[5:-4]] = np.load(f).astype(int)
for f in sorted(glob.glob(os.path.join(P, "ptq_*.npy"))):
    models["ptq_" + os.path.basename(f)[4:-4]] = np.load(f).astype(int)
for tag, d in {"scratch_0.30": "models", "kd_0.30": "models_kd", "pretrained_1.00": "models_pretrained"}.items():
    f = os.path.join(ROOT, d, "qat_preds.npy")
    if os.path.exists(f):
        g = np.load(os.path.join(ROOT, d, "qat_gts.npy")).astype(int)
        assert (g == gts).all(), f"{d}: QAT gts differ from validation order"
        models["qat_" + tag] = np.load(f).astype(int)

rng = np.random.default_rng(SEED)
W = rng.multinomial(n, np.full(n, 1.0 / n), size=B).astype(np.float64)    # (B, n) resample counts

def stats(pred):
    """Return (point F1, point acc, boot F1[B], boot acc[B]); F1 = macro over 6 classes, zero_division=0 (sklearn-equal)."""
    def f1_acc(w):
        w = np.atleast_2d(w)
        f1 = np.zeros(len(w))
        for c in range(NC):
            tp = w @ ((gts == c) & (pred == c)).astype(float)
            tr = w @ (gts == c).astype(float)
            pr = w @ (pred == c).astype(float)
            den = tr + pr                                   # 2tp/(2tp+fp+fn) = 2tp/(true+pred)
            f1 += np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)
        return f1 / NC * 100, (w @ (pred == gts).astype(float)) / w.sum(1) * 100
    pf, pa = f1_acc(np.ones(n))
    bf, ba = f1_acc(W)
    return float(pf[0]), float(pa[0]), bf, ba

S = {k: stats(v) for k, v in models.items()}
ci = lambda x: [float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))]

# sanity: point estimates must equal the saved fp32 results
ref = json.load(open(os.path.join(REV, "fp32_sweep.json")))
for k, v in ref.items():
    key = "fp32_" + k
    if key in S: assert abs(S[key][0] - v["f1_macro"] * 100) < 1e-6, (key, S[key][0], v["f1_macro"] * 100)

out = {"B": B, "seed": SEED, "n": n, "models": {}, "pairs": {}}
for k, (pf, pa, bf, ba) in S.items():
    out["models"][k] = {"f1": pf, "f1_ci95": ci(bf), "f1_boot_sd": float(bf.std(ddof=1)),
                        "acc": pa, "acc_ci95": ci(ba)}

def pair(name, a, b):
    """difference a - b (percentage points)."""
    if a not in S or b not in S: return
    d_f1, d_acc = S[a][2] - S[b][2], S[a][3] - S[b][3]
    out["pairs"][name] = {"a": a, "b": b, "dF1": S[a][0] - S[b][0], "dF1_ci95": ci(d_f1),
                          "dAcc": S[a][1] - S[b][1], "dAcc_ci95": ci(d_acc),
                          "p_a_not_better": float(((d_f1 <= 0).mean()))}

pair("Pretraining effect (width 1.0): pretrained - scratch", "fp32_pretrained_1.00", "fp32_scratch_1.00")
pair("Distillation effect (width 0.3): KD - scratch", "fp32_kd_0.30", "fp32_scratch_0.30")
pair("Remaining gap to teacher: teacher - KD 0.3", "fp32_teacher_1.00", "fp32_kd_0.30")
for w in ("0.25", "0.50", "0.75"):
    pair(f"KD effect at width {w}: KD - scratch", f"fp32_kd_{w}", f"fp32_scratch_{w}")
for a, b in [("0.30", "0.25"), ("0.50", "0.30"), ("0.75", "0.50"), ("1.00", "0.75")]:
    pair(f"Width step scratch {b} -> {a}", f"fp32_scratch_{a}", f"fp32_scratch_{b}")
for t in ("scratch_0.30", "kd_0.30", "pretrained_1.00"):
    pair(f"QAT - fp32 ({t})", f"qat_{t}", f"fp32_{t}")
    pair(f"PTQ - fp32 ({t})", f"ptq_{t}", f"fp32_{t}")
    pair(f"QAT - PTQ ({t})", f"qat_{t}", f"ptq_{t}")

json.dump(out, open(os.path.join(REV, "bootstrap_results.json"), "w"), indent=2)

L = [f"# Bootstrap summary (B={B:,}, n={n}, percentile 95% CI, paired resamples)\n",
     "Evaluation-set noise only; training-seed variance is NOT included.\n",
     "## Per-model scores\n", "| model | F1 % [95% CI] | boot SD | acc % [95% CI] |", "|---|---|---|---|"]
for k, v in out["models"].items():
    L.append(f"| {k} | {v['f1']:.2f} [{v['f1_ci95'][0]:.2f}, {v['f1_ci95'][1]:.2f}] | {v['f1_boot_sd']:.2f} | "
             f"{v['acc']:.2f} [{v['acc_ci95'][0]:.2f}, {v['acc_ci95'][1]:.2f}] |")
L += ["\n## Paired differences (a - b, percentage points)\n",
      "| comparison | dF1 [95% CI] | resolved? | dAcc [95% CI] |", "|---|---|---|---|"]
for k, v in out["pairs"].items():
    lo, hi = v["dF1_ci95"]
    res = "yes" if lo > 0 or hi < 0 else "NO (CI spans 0)"
    L.append(f"| {k} | {v['dF1']:+.2f} [{lo:+.2f}, {hi:+.2f}] | {res} | {v['dAcc']:+.2f} [{v['dAcc_ci95'][0]:+.2f}, {v['dAcc_ci95'][1]:+.2f}] |")
open(os.path.join(REV, "bootstrap_summary.md"), "w").write("\n".join(L) + "\n")
print("\n".join(L))

# Bootstrap summary (B=10,000, n=433, percentile 95% CI, paired resamples)

Evaluation-set noise only; training-seed variance is NOT included.

## Per-model scores

| model | F1 % [95% CI] | boot SD | acc % [95% CI] |
|---|---|---|---|
| fp32_kd_0.25 | 57.43 [52.26, 61.84] | 2.44 | 42.96 [38.34, 47.81] |
| fp32_kd_0.30 | 69.78 [65.34, 73.64] | 2.12 | 57.51 [52.66, 62.12] |
| fp32_kd_0.50 | 73.00 [68.77, 76.66] | 2.01 | 62.12 [57.51, 66.51] |
| fp32_kd_0.75 | 81.33 [77.78, 84.48] | 1.72 | 72.98 [68.82, 77.14] |
| fp32_pretrained_1.00 | 86.92 [83.99, 89.53] | 1.42 | 79.68 [75.75, 83.37] |
| fp32_scratch_0.25 | 19.04 [14.50, 23.31] | 2.23 | 12.70 [9.47, 15.94] |
| fp32_scratch_0.30 | 25.32 [20.54, 29.81] | 2.36 | 18.48 [14.78, 22.17] |
| fp32_scratch_0.50 | 25.87 [20.92, 30.55] | 2.45 | 17.55 [14.09, 21.25] |
| fp32_scratch_0.75 | 36.48 [31.42, 41.11] | 2.48 | 27.94 [23.79, 32.10] |
| fp32_scratch_1.00 | 44.58 [39.34, 49.19] | 2.51 | 33.26 [28.87, 37.64] |
| fp32_teacher_1.00 | 86.92 [83.

In [22]:
res = json.load(open(os.path.join(REV, "bootstrap_results.json")))
print("Per-model bootstrap SD of F1 (pp):", {k: round(v["f1_boot_sd"], 2) for k, v in res["models"].items()})
print("\nPairs whose 95% CI on dF1 spans 0 (cannot be separated from no difference):")
for k, v in res["pairs"].items():
    lo, hi = v["dF1_ci95"]
    if lo <= 0 <= hi:
        print(f"  {k}: {v['dF1']:+.2f} [{lo:+.2f}, {hi:+.2f}]")
print("\nPairs that ARE resolved:")
for k, v in res["pairs"].items():
    lo, hi = v["dF1_ci95"]
    if not (lo <= 0 <= hi):
        print(f"  {k}: {v['dF1']:+.2f} [{lo:+.2f}, {hi:+.2f}]")

Per-model bootstrap SD of F1 (pp): {'fp32_kd_0.25': 2.44, 'fp32_kd_0.30': 2.12, 'fp32_kd_0.50': 2.01, 'fp32_kd_0.75': 1.72, 'fp32_pretrained_1.00': 1.42, 'fp32_scratch_0.25': 2.23, 'fp32_scratch_0.30': 2.36, 'fp32_scratch_0.50': 2.45, 'fp32_scratch_0.75': 2.48, 'fp32_scratch_1.00': 2.51, 'fp32_teacher_1.00': 1.42, 'ptq_kd_0.30': 2.17, 'ptq_pretrained_1.00': 1.73, 'ptq_scratch_0.30': 2.23, 'qat_scratch_0.30': 2.32, 'qat_kd_0.30': 2.13, 'qat_pretrained_1.00': 1.55}

Pairs whose 95% CI on dF1 spans 0 (cannot be separated from no difference):
  Width step scratch 0.25 -> 0.30: +6.28 [-0.22, +12.74]
  Width step scratch 0.30 -> 0.50: +0.55 [-5.41, +6.43]
  QAT - fp32 (scratch_0.30): -1.10 [-5.55, +3.43]
  QAT - fp32 (kd_0.30): -0.59 [-3.13, +1.93]
  PTQ - fp32 (kd_0.30): -1.90 [-4.44, +0.58]
  QAT - PTQ (kd_0.30): +1.32 [-0.40, +3.08]

Pairs that ARE resolved:
  Pretraining effect (width 1.0): pretrained - scratch: +42.35 [+37.29, +47.88]
  Distillation effect (width 0.3): KD - scratch: +44

In [14]:
import json, torch
from torch.utils.data import DataLoader
SEEDS_EXTRA = [1, 2]
SEEDS_DIR = os.path.join(PROJECT_ROOT, "models_seeds"); os.makedirs(SEEDS_DIR, exist_ok=True)
fp = json.load(open(os.path.join(REV_DIR, "split_fingerprint.json")))
assert fp["val_label_md5"] == "d210029e2814357ca28251390535186e", "split changed!"
assert (len(train_loader.dataset), len(val_loader.dataset)) == (3906, 433)
print("split OK: 3906/433, seed-0 fingerprint holds")
def fresh_train_loader(seed, batch=BATCH_SIZE):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(train_loader.dataset, batch_size=batch, shuffle=True, generator=g)

split OK: 3906/433, seed-0 fingerprint holds


In [15]:
SCRATCH_SEED_EPOCHS = 120
for seed in SEEDS_EXTRA:
    ckpt = os.path.join(REV_DIR, f"seed_scratch030_s{seed}.json")
    if os.path.exists(ckpt):
        print(f"seed {seed}: cached, skipping train"); continue
    set_seed(seed)  # init + batch order only; split never rebuilt
    tl = fresh_train_loader(seed)
    m = SSDLiteGestureDetector(width_mult=0.30).to(device)
    _old = MODELS_DIR
    MODELS_DIR = os.path.join(SEEDS_DIR, f"scratch_0.30_s{seed}"); os.makedirs(MODELS_DIR, exist_ok=True)
    try:
        hist, best_ep, best_f1 = train_model(m, tl, val_loader, anchors_center_d, anchors_corner_d, device,
            epochs=SCRATCH_SEED_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_name="best_model.pt")
    finally:
        MODELS_DIR = _old
    met = evaluate(m, val_loader, anchors_center_d, device)
    m.eval(); P = []
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            P.append(predict_batch_labels(m, imgs.to(device), anchors_center_d).cpu())
    P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
    np.save(os.path.join(REV_DIR, "preds", f"seed_scratch030_s{seed}.npy"), P)
    json.dump({"seed": seed, "kind": "scratch", "width": 0.30, "best_epoch": best_ep,
               "fp32": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")},
               "n_missed": met["n_missed"]}, open(ckpt, "w"), indent=2)
    print(f"seed {seed} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} acc={met['accuracy']*100:.2f} -> {ckpt}")

backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width


training:   0%|          | 0/120 [00:00<?, ?epoch/s]

epoch 1/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]

epoch  1/120  total=12.9701  loc=6.8598  cls=6.1103  |  val_acc=0.000  val_prec=0.000  val_rec=0.000  val_f1=0.000  (missed 433/433)  <-- best so far


...

epoch 120/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]

epoch 120/120  total=2.8909  loc=1.1021  cls=1.7888  |  val_acc=0.453  val_prec=0.896  val_rec=0.451  val_f1=0.591  (missed 213/433)

best checkpoint: epoch 97  f1_macro=0.5978
loaded best checkpoint into the passed-in model and saved it to c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds\scratch_0.30_s2\best_model.pt


validating:   0%|          | 0/7 [00:00<?, ?it/s]

seed 2 done: best_ep=97 F1=59.78 acc=46.19 -> c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\revision_results\seed_scratch030_s2.json


In [16]:
# Fresh seed-0 baseline with the CURRENT pipeline (stale-artifact test)
seed = 0
ckpt = os.path.join(REV_DIR, "seed_scratch030_s0.json")
assert not os.path.exists(ckpt), f"{ckpt} exists -- delete it to redo seed 0"
set_seed(seed)  # same call S2 uses; split untouched (S1 guard already asserted it)
tl = fresh_train_loader(seed)
m = SSDLiteGestureDetector(width_mult=0.30).to(device)
_old = MODELS_DIR
MODELS_DIR = os.path.join(SEEDS_DIR, "scratch_0.30_s0"); os.makedirs(MODELS_DIR, exist_ok=True)
try:
    hist, best_ep, best_f1 = train_model(m, tl, val_loader, anchors_center_d, anchors_corner_d, device,
        epochs=120, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_name="best_model.pt")
finally:
    MODELS_DIR = _old
met = evaluate(m, val_loader, anchors_center_d, device)
m.eval(); P = []
with torch.no_grad():
    for imgs, _, _ in val_loader:
        P.append(predict_batch_labels(m, imgs.to(device), anchors_center_d).cpu())
P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
np.save(os.path.join(REV_DIR, "preds", "seed_scratch030_s0.npy"), P)
json.dump({"seed": seed, "kind": "scratch", "width": 0.30, "best_epoch": best_ep,
           "fp32": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")},
           "n_missed": met["n_missed"]}, open(ckpt, "w"), indent=2)
print(f"seed {seed} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} acc={met['accuracy']*100:.2f} -> {ckpt}")

backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width


training:   0%|          | 0/120 [00:00<?, ?epoch/s]

epoch 1/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]

epoch  1/120  total=13.5362  loc=6.9043  cls=6.6319  |  val_acc=0.000  val_prec=0.000  val_rec=0.000  val_f1=0.000  (missed 433/433)  <-- best so far


...

epoch 120/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]

epoch 120/120  total=2.9130  loc=1.0872  cls=1.8258  |  val_acc=0.450  val_prec=0.852  val_rec=0.449  val_f1=0.584  (missed 206/433)

best checkpoint: epoch 74  f1_macro=0.6081
loaded best checkpoint into the passed-in model and saved it to c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds\scratch_0.30_s0\best_model.pt


validating:   0%|          | 0/7 [00:00<?, ?it/s]

seed 0 done: best_ep=74 F1=60.81 acc=48.27 -> c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\revision_results\seed_scratch030_s0.json


In [17]:
img, box, label = val_loader.dataset[0]
print(type(img), tuple(img.shape), img.min().item(), img.max().item())
print("box:", box, "label:", label, CLASS_NAMES[label])
print("anchors_corner[0]:", anchors_corner[0])

<class 'torch.Tensor'> (3, 128, 128) 0.0 1.0
box: tensor([0.6636, 0.1464, 0.8114, 0.4354]) label: tensor(4) peace
anchors_corner[0]: tensor([ 0.0120, -0.0256,  0.1130,  0.1506])


In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

P = np.load(os.path.join(REV_DIR, "preds", "seed_scratch030_s0.npy"))  # or fp32_kd_0.30.npy etc.
G = np.load(os.path.join(REV_DIR, "preds", "gts.npy"))
m = SSDLiteGestureDetector(width_mult=0.30).to(device)
m.load_state_dict(torch.load(os.path.join(SEEDS_DIR, "scratch_0.30_s0", "best_model.pt"), map_location=device)); m.eval()

def draw(idx):
    img, box, label = val_loader.dataset[idx]
    with torch.no_grad():
        cls_preds, loc_preds = m(img.unsqueeze(0).to(device))
    probs = F.softmax(cls_preds, dim=-1)[..., 1:]
    score, c = probs.max(dim=-1)
    score = score.masked_fill(score < 0.5, -1.0)
    top, aidx = score.max(dim=-1)
    a = aidx.item()
    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    ax.imshow(img.permute(1, 2, 0).cpu())
    ax, ay, aw, ah = (anchors_corner[a].numpy() * IMG_SIZE)
    ax.add_patch(plt.Rectangle((ax, ay), aw, ah,  # top anchor
                fill=False, edgecolor="red", lw=2, ls="--"))

# misses first, then the ok/palm confusion the paper flags:
misses = np.where(P != G)[0]
print("n misses:", len(misses))
for i in misses[:6]: draw(int(i))
ok = [i for i in misses if G[i] == CLASS_NAMES.index("ok")][:4]
for i in ok: draw(int(i))

IndentationError: unexpected indent (1307513502.py, line 20)

In [54]:
# Dynamo-free optimizers: strip torch 2.5's _disable_dynamo wrappers (they import
# torch._dynamo on first optimizer construction). Eager training doesn't need dynamo.
import torch.optim.optimizer as _OO
for _cls in (torch.optim.SGD, torch.optim.AdamW):
    for _m in ("add_param_group", "step", "zero_grad"):
        _f = getattr(getattr(_cls, _m, None), "__wrapped__", None)
        if _f is not None and getattr(_cls, _m) is not _f:
            setattr(_cls, _m, torch.no_grad()(_f))
print("optimizers dynamo-free")

optimizers dynamo-free


In [59]:
# Q1 — dynamo purge + strip, then QAT sweep (40ep, fixed-epoch, NO best-selection)
import sys
for m in [m for m in sys.modules if m == "torch._dynamo" or m.startswith("torch._dynamo.")]:
    del sys.modules[m]
import torch._dynamo
print("dynamo ok")

def _strip(fn):
    seen = 0
    while hasattr(fn, "__wrapped__") and seen < 10:
        fn = fn.__wrapped__
        seen += 1
    return torch.no_grad()(fn)

import torch.optim.optimizer as _OO
for _cls in (torch.optim.SGD, torch.optim.AdamW):
    for _m in ("add_param_group", "step", "zero_grad"):
        _raw = _strip(getattr(_cls, _m))
        if getattr(_cls, _m) is not _raw:
            setattr(_cls, _m, _raw)
print("optimizers stripped")

QAT_SWEEP_EPOCHS = 40
QAT_SWEEP_JOBS = {"kd_0.30": "models_kd/gesture_ssdlite.onnx",
                  "pretrained_1.00": "models_pretrained/gesture_ssdlite.onnx"}
for key, onnx_p in QAT_SWEEP_JOBS.items():
    out = os.path.join(REV_DIR, "ptq_no_test", f"qat_sweep_{key}.espdl")
    calib = DataLoader(val_loader.dataset, batch_size=1, shuffle=False)
    graph = espdl_quantize_onnx(
        onnx_import_file=onnx_p, espdl_export_file=out,
        calib_dataloader=calib, calib_steps=min(200, len(val_loader.dataset)),
        input_shape=[1, 3, IMG_SIZE, IMG_SIZE], inputs=None, target=ESPDL_TARGET,
        num_of_bits=ESPDL_NUM_BITS, collate_fn=collate_fn, device=ESPDL_DEVICE,
        error_report=False, skip_export=False, export_test_values=False, verbose=0)
    tr = QATTrainer(graph, device=ESPDL_DEVICE, accum_steps=BATCH_SIZE)
    ql = DataLoader(train_loader.dataset, batch_size=1, shuffle=True,
                    generator=torch.Generator().manual_seed(0))
    rep = {"model": key}
    for ep in range(1, QAT_SWEEP_EPOCHS + 1):
        loss = tr.epoch(ql, anchors_center_d, anchors_corner_d)
        met, _, _ = tr.eval(val_loader.dataset, anchors_center_d)
        rep[ep] = {"loss": loss, "f1": float(met["f1_macro"])*100, "acc": float(met["accuracy"])*100}
        print(f"{key} QAT ep {ep}/{QAT_SWEEP_EPOCHS} loss={loss:.4f} F1={rep[ep]['f1']:.2f}", flush=True)
    json.dump(rep, open(os.path.join(REV_DIR, f"qat_sweep_{key}.json"), "w"), indent=2)
print("fixed epochs (no best-selection):")
for key in QAT_SWEEP_JOBS:
    r = json.load(open(os.path.join(REV_DIR, f"qat_sweep_{key}.json")))
    print(key, " | ".join(f"ep{e}: F1={r[str(e)]['f1']:.2f}" for e in (10, 20, 40)))

dynamo ok
optimizers stripped
[16:36:52] ConvTranspose Decomposition Pass Running ... Finished.
[16:36:53] PPQ Quantization Fusion Pass Running ...       Finished.
[16:36:54] PPQ Quantize Simplify Pass Running ...         Finished.
[16:36:54] PPQ Parameter Quantization Pass Running ...    Finished.
[16:36:54] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 200/200 [00:19<00:00, 10.29it/s]


Finished.
[16:37:40] PPQ Quantization Alignment Pass Running ...    Finished.
[16:37:40] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [85]
Num of Quantized Op:          [85]
Num of Variable:              [232]
Num of Quantized Var:         [232]
------- Quantization Snapshot ------
Num of Quant Config:          [324]
ACTIVATED:                    [87]
OVERLAPPED:                   [110]
PASSIVE:                      [123]
FP32:                         [4]
Network Quantization Finished.
[INFO][ESPDL][2026-10-06 16:37:40]:  Skip /m/Constant_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 16:37:40]:  Skip /m/Constant_2_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 16:37:40]:  Skip /m/Constant_1_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 16:37:40]:  Skip /m/Constant_3_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 16:37:41]:  Skip /m/Constant_output

QAT epoch 1/10:   0%|          | 0/3906 [00:00<?, ?it/s]

kd_0.30 QAT ep 1/40 loss=2.9993 F1=67.58


QAT epoch 2/10:   0%|          | 0/3906 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [62]:
import copy

class ONNXWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, x):
        cls_out, loc_out = self.m(x)
        return cls_out, loc_out

def export_to_onnx(model, onnx_path):
    export_model = ONNXWrapper(copy.deepcopy(model)).to("cpu").eval()
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    torch.onnx.export(
        export_model, dummy_input, onnx_path,
        opset_version=13,
        input_names=["input"],
        output_names=["cls_logits", "loc_preds"],
        do_constant_folding=True,
        dynamic_axes=None,
        dynamo=False,
    )
    print("exported:", onnx_path, f"{os.path.getsize(onnx_path)/1024:.1f} KB")

for s in (0, 1, 2):
    d = os.path.join(SEEDS_DIR, f"scratch_0.30_s{s}")
    out = os.path.join(d, "gesture_ssdlite.onnx")
    if os.path.exists(out):
        print(out, "cached"); continue
    m = SSDLiteGestureDetector(width_mult=0.30).to("cpu")
    m.load_state_dict(torch.load(os.path.join(d, "best_model.pt"), map_location="cpu"))
    export_to_onnx(m, out)

backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width


C:\Users\xared\AppData\Local\Temp\ipykernel_22396\371493010.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(os.path.join(d, "best_model.pt"

exported: c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds\scratch_0.30_s0\gesture_ssdlite.onnx 360.3 KB
backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width
exported: c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds\scratch_0.30_s1\gesture_ssdlite.onnx 360.3 KB
backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width
exported: c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds\scratch_0.30_s2\gesture_ssdlite.onnx 360.3 KB


In [64]:
# M5 — PTQ with/without test values, 3 fresh 0.3 seeds (s0-notest skips itself)
PTQ_JOBS_ABL = {
    "seed_scratch030_s0": os.path.join(SEEDS_DIR, "scratch_0.30_s0", "gesture_ssdlite.onnx"),
    "seed_scratch030_s1": os.path.join(SEEDS_DIR, "scratch_0.30_s1", "gesture_ssdlite.onnx"),
    "seed_scratch030_s2": os.path.join(SEEDS_DIR, "scratch_0.30_s2", "gesture_ssdlite.onnx"),
}
for key, onnx_p in PTQ_JOBS_ABL.items():
    assert os.path.exists(onnx_p), f"export {onnx_p} first (Section-10 cell, static batch=1)"
    for tv, suf in ((False, "notest"), (True, "test")):
        out = os.path.join(REV_DIR, "ptq_no_test", f"abl_ptq_{key}_{suf}.espdl")
        if os.path.exists(out):
            print(f"{key} {suf} cached: {os.path.getsize(out)/1024:.2f}KB", flush=True)
            continue
        calib = DataLoader(val_loader.dataset, batch_size=1, shuffle=False)
        graph = espdl_quantize_onnx(
            onnx_import_file=onnx_p, espdl_export_file=out,
            calib_dataloader=calib, calib_steps=min(200, len(val_loader.dataset)),
            input_shape=[1, 3, IMG_SIZE, IMG_SIZE], inputs=None, target=ESPDL_TARGET,
            num_of_bits=ESPDL_NUM_BITS, collate_fn=collate_fn, device=ESPDL_DEVICE,
            error_report=False, skip_export=False, export_test_values=tv, verbose=0)
        if not tv:
            met, preds, _ = evaluate_espdl(TorchExecutor(graph=graph, device=ESPDL_DEVICE),
                                           val_loader.dataset, anchors_center_d, ESPDL_DEVICE)
            np.save(os.path.join(REV_DIR, "preds", f"abl_ptq_{key}.npy"), preds)
            json.dump({"ptq": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")}},
                      open(os.path.join(REV_DIR, f"abl_ptq_{key}.json"), "w"), indent=2)
            print(f"{key}: F1={met['f1_macro']*100:.2f}", end=" ", flush=True)
        print(f"{key} {suf}={os.path.getsize(out)/1024:.2f}KB", flush=True)

seed_scratch030_s0 notest cached: 183.98KB


seed_scratch030_s0 test cached: 233.66KB
seed_scratch030_s1 notest cached: 183.31KB
seed_scratch030_s1 test cached: 232.98KB
seed_scratch030_s2 notest cached: 183.98KB
[20:07:48] ConvTranspose Decomposition Pass Running ... Finished.
[20:07:49] PPQ Quantization Fusion Pass Running ...       Finished.
[20:07:50] PPQ Quantize Simplify Pass Running ...         Finished.
[20:07:50] PPQ Parameter Quantization Pass Running ...    Finished.
[20:07:50] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 200/200 [00:18<00:00, 10.53it/s]


Finished.
[20:08:29] PPQ Quantization Alignment Pass Running ...    Finished.
[20:08:29] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [85]
Num of Quantized Op:          [85]
Num of Variable:              [232]
Num of Quantized Var:         [232]
------- Quantization Snapshot ------
Num of Quant Config:          [324]
ACTIVATED:                    [87]
OVERLAPPED:                   [110]
PASSIVE:                      [123]
FP32:                         [4]
Network Quantization Finished.
[INFO][ESPDL][2026-10-06 20:08:30]:  Skip /m/Constant_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 20:08:30]:  Skip /m/Constant_2_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 20:08:30]:  Skip /m/Constant_1_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 20:08:30]:  Skip /m/Constant_3_output_0 because it's not exportable
[INFO][ESPDL][2026-10-06 20:08:30]:  Skip /m/Constant_output

In [39]:
# M1 — paired loaders (SAME split, only augmentation toggles)
import json, torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
fp = json.load(open(os.path.join(REV_DIR, "split_fingerprint.json")))
assert fp["val_label_md5"] == "d210029e2814357ca28251390535186e", "split changed!"
_final = load_from_disk(DATASET_CACHE_DIR).shuffle(seed=SEED)
_nv = max(1, int(0.1 * len(_final)))
_hf_val, _hf_train = _final.select(range(_nv)), _final.select(range(_nv, len(_final)))
TRDS_ON = HaGRIDGestureDataset(_hf_train, hagrid_label_names, augment=True)
TRDS_OFF = HaGRIDGestureDataset(_hf_train, hagrid_label_names, augment=False)
assert (np.array([int(l) for _, _, l in HaGRIDGestureDataset(
    _hf_val, hagrid_label_names, augment=False)]) == np.load(
    os.path.join(REV_DIR, "preds", "gts.npy"))).all(), "val drifted!"
def fresh_loader_for(ds, seed, batch=BATCH_SIZE):
    return DataLoader(ds, batch_size=batch, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))
print(f"paired ready: train={len(_hf_train)} val={len(_hf_val)} (aug toggles ONLY in train)")

paired ready: train=3906 val=433 (aug toggles ONLY in train)


In [40]:
# M2 — unified scratch trainer
def train_job(kind, width, seed, aug_tag, epochs=120, force_random_init=False):
    global MODELS_DIR  # train_model saves into this global; without this the assignment below breaks
    tag = f"{kind}_{width:.2f}_{aug_tag}_s{seed}".replace(".", "p")
    ckpt = os.path.join(REV_DIR, f"abl_{tag}.json")
    if os.path.exists(ckpt):
        print(tag, "cached, skipping"); return json.load(open(ckpt))
    set_seed(seed)
    ds = TRDS_ON if aug_tag == "aug" else TRDS_OFF
    tl = fresh_loader_for(ds, seed)
    m = SSDLiteGestureDetector(width_mult=width).to(device)
    if force_random_init:
        # builder hardcodes ImageNet weights at width 1.0: wipe to seeded random init for TRUE scratch
        m.apply(lambda mod: mod.reset_parameters() if isinstance(mod, (nn.Conv2d, nn.Linear, nn.BatchNorm2d)) else None)
        print("forced random init (true from-scratch at width 1.0)")
    _old = MODELS_DIR
    MODELS_DIR = os.path.join(SEEDS_DIR, f"abl_{tag}"); os.makedirs(MODELS_DIR, exist_ok=True)
    try:
        hist, best_ep, best_f1 = train_model(m, tl, val_loader, anchors_center_d, anchors_corner_d,
            device, epochs=epochs, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_name="best_model.pt")
    finally:
        MODELS_DIR = _old
    met = evaluate(m, val_loader, anchors_center_d, device)
    m.eval(); P = []
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            P.append(predict_batch_labels(m, imgs.to(device), anchors_center_d).cpu())
    P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
    np.save(os.path.join(REV_DIR, "preds", f"abl_{tag}.npy"), P)
    rec = {"kind": kind, "width": width, "aug": aug_tag, "seed": seed, "best_epoch": best_ep,
           "force_random_init": force_random_init,
           "params": sum(p.numel() for p in m.parameters()),
           "fp32": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")},
           "n_missed": met["n_missed"]}
    json.dump(rec, open(ckpt, "w"), indent=2)
    print(f"{tag} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} -> {ckpt}")
    return rec

In [41]:
class DistillationLoss(nn.Module):
    """
    Hard-label MultiBoxLoss (ground truth, unchanged) plus a soft distillation term pulling
    the student toward a frozen teacher's outputs.
      - classification: KL divergence between temperature-softened student/teacher class
        distributions (Hinton et al. 2015 soft-target distillation), weighted by `alpha`.
      - localization: SmoothL1 between student and teacher box regression outputs, at the
        same positive anchors the hard loss already uses -- soft box targets only mean
        something where there's an actual box to regress -- weighted by `beta`.
    """

    def __init__(self, neg_pos_ratio: int = NEG_POS_RATIO, temperature: float = KD_TEMPERATURE,
                 alpha: float = KD_ALPHA, beta: float = KD_BETA):
        super().__init__()
        self.hard_loss = MultiBoxLoss(neg_pos_ratio=neg_pos_ratio)
        self.temperature = temperature
        self.alpha = alpha
        self.beta = beta

    def forward(self, student_cls, student_loc, teacher_cls, teacher_loc,
                cls_targets, loc_targets, pos_mask, ignore_mask):
        hard_total, hard_loc, hard_cls = self.hard_loss(
            student_cls, student_loc, cls_targets, loc_targets, pos_mask, ignore_mask)

        T = self.temperature
        student_log_soft = F.log_softmax(student_cls / T, dim=-1)
        teacher_soft = F.softmax(teacher_cls / T, dim=-1).detach()
        soft_cls = F.kl_div(student_log_soft, teacher_soft, reduction="batchmean") * (T ** 2)

        num_pos = pos_mask.sum().clamp(min=1).float()
        soft_loc = F.smooth_l1_loss(student_loc[pos_mask], teacher_loc[pos_mask].detach(),
                                     reduction="sum") / num_pos

        total = hard_total + self.alpha * soft_cls + self.beta * soft_loc
        return total, hard_loc, hard_cls, soft_cls, soft_loc


def train_model_kd(student, teacher, train_loader, val_loader, anchors_center, anchors_corner, device,
                    epochs: int = STUDENT_EPOCHS, lr: float = LR, weight_decay: float = WEIGHT_DECAY,
                    selection_metric: str = SELECTION_METRIC, checkpoint_name: str = "best_model.pt",
                    temperature: float = KD_TEMPERATURE, alpha: float = KD_ALPHA, beta: float = KD_BETA):
    """
    Same shape as train_model, but the student is also pulled toward a frozen teacher's
    outputs via DistillationLoss. Only `student`'s parameters get gradient updates -- the
    teacher is frozen and always run in eval mode under no_grad.
    """
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    criterion = DistillationLoss(neg_pos_ratio=NEG_POS_RATIO, temperature=temperature,
                                   alpha=alpha, beta=beta).to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {
        "total": [], "loc": [], "cls": [], "soft_cls": [], "soft_loc": [],
        "val_accuracy": [], "val_precision": [], "val_recall": [], "val_f1_macro": [],
        "iter_total": [], "iter_loc": [], "iter_cls": [], "iter_soft_cls": [], "iter_soft_loc": [],
    }

    best_metric_value = -1.0
    best_state_dict = None
    best_epoch = -1

    epoch_bar = tqdm(range(1, epochs + 1), desc="distilling", unit="epoch")
    for epoch in epoch_bar:
        student.train()
        running = {"total": 0.0, "loc": 0.0, "cls": 0.0, "soft_cls": 0.0, "soft_loc": 0.0}
        n_batches = 0

        batch_bar = tqdm(train_loader, desc=f"epoch {epoch}/{epochs}", leave=False)
        for imgs, boxes, labels in batch_bar:
            imgs, boxes, labels = imgs.to(device), boxes.to(device), labels.to(device)

            with torch.no_grad():
                teacher_cls, teacher_loc = teacher(imgs)

            student_cls, student_loc = student(imgs)
            cls_t, loc_t, pos_m, ign_m = match_anchors_to_targets(boxes, labels, anchors_center, anchors_corner)
            total_loss, hard_loc, hard_cls, soft_cls, soft_loc = criterion(
                student_cls, student_loc, teacher_cls, teacher_loc, cls_t, loc_t, pos_m, ign_m)

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=5.0)
            optimizer.step()

            history["iter_total"].append(total_loss.item())
            history["iter_loc"].append(hard_loc.item())
            history["iter_cls"].append(hard_cls.item())
            history["iter_soft_cls"].append(soft_cls.item())
            history["iter_soft_loc"].append(soft_loc.item())

            running["total"] += total_loss.item()
            running["loc"] += hard_loc.item()
            running["cls"] += hard_cls.item()
            running["soft_cls"] += soft_cls.item()
            running["soft_loc"] += soft_loc.item()
            n_batches += 1

            batch_bar.set_postfix(total=running["total"] / n_batches,
                                   hard_cls=running["cls"] / n_batches,
                                   soft_cls=running["soft_cls"] / n_batches)

        scheduler.step()
        for k in ("total", "loc", "cls", "soft_cls", "soft_loc"):
            history[k].append(running[k] / n_batches)

        val_metrics = evaluate(student, val_loader, anchors_center, device)
        for k in ("accuracy", "precision", "recall", "f1_macro"):
            history[f"val_{k}"].append(val_metrics[k])

        is_best = val_metrics[selection_metric] > best_metric_value
        if is_best:
            best_metric_value = val_metrics[selection_metric]
            best_state_dict = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            best_epoch = epoch

        epoch_bar.set_postfix(total=f"{history['total'][-1]:.3f}",
                               val_f1=f"{val_metrics['f1_macro']:.3f}",
                               best=f"{best_metric_value:.3f}@{best_epoch}")

        print(f"epoch {epoch:2d}/{epochs}  total={history['total'][-1]:.4f}  "
              f"hard_loc={history['loc'][-1]:.4f}  hard_cls={history['cls'][-1]:.4f}  "
              f"soft_cls={history['soft_cls'][-1]:.4f}  soft_loc={history['soft_loc'][-1]:.4f}  |  "
              f"val_acc={val_metrics['accuracy']:.3f}  val_prec={val_metrics['precision']:.3f}  "
              f"val_rec={val_metrics['recall']:.3f}  val_f1={val_metrics['f1_macro']:.3f}  "
              f"(missed {val_metrics['n_missed']}/{val_metrics['n_total']})"
              f"{'  <-- best so far' if is_best else ''}")

    print(f"\nbest checkpoint: epoch {best_epoch}  {selection_metric}={best_metric_value:.4f}")
    best_model_path = os.path.join(MODELS_DIR, checkpoint_name)
    torch.save(best_state_dict, best_model_path)
    student.load_state_dict(best_state_dict)
    print(f"loaded best checkpoint into `student_model` and saved it to {best_model_path}")

    return history, best_epoch, best_metric_value


# --- scaffolding only (not yours): per-seed driver around the verbatim loop above ---
def kd_student_job(seed, epochs=STUDENT_EPOCHS):
    global MODELS_DIR
    tag = f"kd_0p30_aug_s{seed}"
    ckpt = os.path.join(REV_DIR, f"abl_{tag}.json")
    if os.path.exists(ckpt):
        print(tag, "cached, skipping"); return json.load(open(ckpt))
    set_seed(seed)
    tl = fresh_loader_for(TRDS_ON, seed)
    tckpt = os.path.join(PROJECT_ROOT, "models_kd", "teacher_best.pt")
    assert os.path.exists(tckpt), f"missing {tckpt}"
    teacher = SSDLiteGestureDetector(width_mult=1.0).to(device)
    teacher.load_state_dict(torch.load(tckpt, map_location=device))
    student = SSDLiteGestureDetector(width_mult=0.30).to(device)
    _old = MODELS_DIR
    MODELS_DIR = os.path.join(SEEDS_DIR, f"abl_{tag}"); os.makedirs(MODELS_DIR, exist_ok=True)
    try:
        hist, best_ep, best_f1 = train_model_kd(student, teacher, tl, val_loader,
            anchors_center_d, anchors_corner_d, device, epochs=epochs,
            checkpoint_name="best_model.pt")
    finally:
        MODELS_DIR = _old
    met = evaluate(student, val_loader, anchors_center_d, device)
    student.eval(); P = []
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            P.append(predict_batch_labels(student, imgs.to(device), anchors_center_d).cpu())
    P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
    np.save(os.path.join(REV_DIR, "preds", f"abl_{tag}.npy"), P)
    rec = {"kind": "kd", "width": 0.3, "aug": "aug", "seed": seed, "teacher": tckpt,
           "best_epoch": best_ep,
           "fp32": {k: float(met[k]) for k in ("accuracy", "precision", "recall", "f1_macro")},
           "n_missed": met["n_missed"]}
    json.dump(rec, open(ckpt, "w"), indent=2)
    print(f"{tag} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} -> {ckpt}")
    return rec

In [45]:
SEEDS_DIR = os.path.join(PROJECT_ROOT, "models_seeds"); os.makedirs(SEEDS_DIR, exist_ok=True)
print("SEEDS_DIR =", SEEDS_DIR)

SEEDS_DIR = c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds


In [46]:
# T-queue: KD s1/s2 + scratch-noaug s1/s2 (~6h, skip-if-cached)
import sys
for m in [m for m in sys.modules if m == "torch._dynamo" or m.startswith("torch._dynamo.")]:
    del sys.modules[m]
import torch._dynamo
print("dynamo ok")
for s in (1, 2):
    try:
        kd_student_job(seed=s)
    except Exception as e:
        print(f"kd s{s} FAILED: {type(e).__name__}: {e}")
for s in (1, 2):
    try:
        train_job("scratch", 0.3, s, "off")
    except Exception as e:
        print(f"noaug s{s} FAILED: {type(e).__name__}: {e}")
print("T done; paste this log back")

dynamo ok
backbone: ImageNet-pretrained MobileNetV2 (width_mult=1.0)


C:\Users\xared\AppData\Local\Temp\ipykernel_23844\3231118937.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(tckpt, map_location=dev

backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width


distilling:   0%|          | 0/120 [00:00<?, ?epoch/s]

epoch 1/120:   0%|          | 0/62 [00:00<?, ?it/s]

...

C:\Users\xared\AppData\Local\Temp\ipykernel_23844\3231118937.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(tckpt, map_location=dev

backbone: from-scratch MobileNetV2 (width_mult=0.3) -- no pretrained checkpoint exists at this width


distilling:   0%|          | 0/120 [00:00<?, ?epoch/s]

epoch 1/120:   0%|          | 0/62 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [47]:
# M5-DEPS + M5 — PTQ with/without test values, 3 fresh 0.3 seeds (self-healing)
from esp_ppq.api import espdl_quantize_onnx
from esp_ppq.executor import TorchExecutor
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import json, os
import numpy as np

ESPDL_TARGET = "esp32s3"
ESPDL_NUM_BITS = 8
ESPDL_CALIB_STEPS = min(200, len(val_loader.dataset))
ESPDL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def collate_fn(batch):
    imgs, _boxes, _labels = batch
    return imgs.to(ESPDL_DEVICE)

@torch.no_grad()
def evaluate_espdl(executor, dataset, anchors_center, device, conf_thresh=0.5):
    all_preds, all_gts = [], []
    for img, box, label in dataset:
        img_in = img.unsqueeze(0).to(device)
        outputs = executor(img_in)
        cls_out, loc_out = outputs[0], outputs[1]
        probs = F.softmax(cls_out.float(), dim=-1)
        fg_probs = probs[..., 1:]
        best_score, best_cls = fg_probs.max(dim=-1)
        thresholded = best_score.masked_fill(best_score < conf_thresh, -1.0)
        top_score, top_anchor = thresholded.max(dim=-1)
        pred_cls = best_cls.gather(1, top_anchor.unsqueeze(1)).squeeze(1)
        pred_label = int(pred_cls.item()) if top_score.item() >= 0 else NUM_CLASSES
        all_preds.append(pred_label)
        all_gts.append(int(label))
    all_preds = np.array(all_preds)
    all_gts = np.array(all_gts)
    accuracy = accuracy_score(all_gts, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_gts, all_preds, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0)
    n_missed = int((all_preds == NUM_CLASSES).sum())
    return ({"accuracy": accuracy, "precision": precision, "recall": recall,
             "f1_macro": f1, "n_missed": n_missed, "n_total": len(all_gts)},
            all_preds, all_gts)

PTQ_JOBS_ABL = {
    "seed_scratch030_s0": os.path.join(SEEDS_DIR, "scratch_0.30_s0", "gesture_ssdlite.onnx"),
    "seed_scratch030_s1": os.path.join(SEEDS_DIR, "scratch_0.30_s1", "gesture_ssdlite.onnx"),
    "seed_scratch030_s2": os.path.join(SEEDS_DIR, "scratch_0.30_s2", "gesture_ssdlite.onnx"),
}
for key, onnx_p in PTQ_JOBS_ABL.items():
    assert os.path.exists(onnx_p), f"export {onnx_p} first (Section-10 cell, static batch=1)"
    jq = os.path.join(REV_DIR, f"abl_ptq_{key}.json")
    npy = os.path.join(REV_DIR, "preds", f"abl_ptq_{key}.npy")
    for tv, suf in ((False, "notest"), (True, "test")):
        out = os.path.join(REV_DIR, "ptq_no_test", f"abl_ptq_{key}_{suf}.espdl")
        complete = os.path.exists(out) and (tv or (os.path.exists(jq) and os.path.exists(npy)))
        if complete:
            print(f"{key} {suf} cached: {os.path.getsize(out)/1024:.2f}KB", flush=True)
            continue
        calib = DataLoader(val_loader.dataset, batch_size=1, shuffle=False)
        graph = espdl_quantize_onnx(
            onnx_import_file=onnx_p, espdl_export_file=out,
            calib_dataloader=calib, calib_steps=min(200, len(val_loader.dataset)),
            input_shape=[1, 3, IMG_SIZE, IMG_SIZE], inputs=None, target=ESPDL_TARGET,
            num_of_bits=ESPDL_NUM_BITS, collate_fn=collate_fn, device=ESPDL_DEVICE,
            error_report=False, skip_export=False, export_test_values=tv, verbose=0)
        if not tv:
            met, preds, _ = evaluate_espdl(TorchExecutor(graph=graph, device=ESPDL_DEVICE),
                                           val_loader.dataset, anchors_center_d, ESPDL_DEVICE)
            np.save(npy, preds)
            json.dump({"ptq": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")}},
                      open(jq, "w"), indent=2)
            print(f"{key}: F1={met['f1_macro']*100:.2f}", end=" ", flush=True)
        print(f"{key} {suf}={os.path.getsize(out)/1024:.2f}KB", flush=True)


    ___________ ____        ____  ____  ____
   / ____/ ___// __ \      / __ \/ __ \/ __ \
  / __/  \__ \/ /_/ /_____/ /_/ / /_/ / / / /
 / /___ ___/ / ____/_____/ ____/ ____/ /_/ /
/_____//____/_/         /_/   /_/    \___\_\


seed_scratch030_s0 notest cached: 183.98KB
seed_scratch030_s0 test cached: 233.66KB
seed_scratch030_s1 notest cached: 183.31KB
seed_scratch030_s1 test cached: 232.98KB
[01:41:25] ConvTranspose Decomposition Pass Running ... Finished.
[01:41:26] PPQ Quantization Fusion Pass Running ...       Finished.
[01:41:26] PPQ Quantize Simplify Pass Running ...         Finished.
[01:41:26] PPQ Parameter Quantization Pass Running ...    Finished.
[01:41:26] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 200/200 [00:16<00:00, 12.26it/s]


Finished.
[01:41:59] PPQ Quantization Alignment Pass Running ...    Finished.
[01:41:59] PPQ Passive Parameter Quantization Running ... Finished.
--------- Network Snapshot ---------
Num of Op:                    [85]
Num of Quantized Op:          [85]
Num of Variable:              [232]
Num of Quantized Var:         [232]
------- Quantization Snapshot ------
Num of Quant Config:          [324]
ACTIVATED:                    [87]
OVERLAPPED:                   [110]
PASSIVE:                      [123]
FP32:                         [4]
Network Quantization Finished.
[INFO][ESPDL][2026-10-07 01:41:59]:  Skip /m/Constant_output_0 because it's not exportable
[INFO][ESPDL][2026-10-07 01:41:59]:  Skip /m/Constant_2_output_0 because it's not exportable
[INFO][ESPDL][2026-10-07 01:41:59]:  Skip /m/Constant_1_output_0 because it's not exportable
[INFO][ESPDL][2026-10-07 01:41:59]:  Skip /m/Constant_3_output_0 because it's not exportable
[INFO][ESPDL][2026-10-07 01:41:59]:  Skip /m/Constant_output

In [ ]:
# A/B: ORIGINAL train_model + ORIGINAL loader build, seed 0, 120ep.
# Arm A (unaug) replicates the paper's 25.32 run. Arm B (aug) replicates 60.81.
import sys
for m in [m for m in sys.modules if m == "torch._dynamo" or m.startswith("torch._dynamo.")]:
    del sys.modules[m]
import torch._dynamo
print("dynamo ok")
import torch.optim.optimizer as _OO
for _cls in (torch.optim.SGD, torch.optim.AdamW):
    for _m in ("add_param_group", "step", "zero_grad"):
        def _strip(fn):
            seen = 0
            while hasattr(fn, "__wrapped__") and seen < 10:
                fn = fn.__wrapped__
                seen += 1
            return torch.no_grad()(fn)
        _raw = _strip(getattr(_cls, _m))
        if getattr(_cls, _m) is not _raw:
            setattr(_cls, _m, _raw)

def og_loader(ds):
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                                       num_workers=0, pin_memory=torch.cuda.is_available())

for tag, aug in (("A_unaug", False), ("B_aug", True)):
    ck = os.path.join(REV_DIR, f"ab_{tag}_s0.json")
    if os.path.exists(ck):
        print(tag, "cached"); continue
    set_seed(0)
    ds = HaGRIDGestureDataset(_hf_train, hagrid_label_names, augment=aug)
    tl = og_loader(ds)
    m = SSDLiteGestureDetector(width_mult=0.30).to(device)
    hist, best_ep, best_f1 = train_model(m, tl, val_loader, anchors_center_d, anchors_corner_d, device,
        epochs=120, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_name=f"ab_{tag}_s0.pt")
    met = evaluate(m, val_loader, anchors_center_d, device)
    json.dump({"arm": tag, "aug": aug, "seed": 0, "best_epoch": best_ep,
               "fp32": {k: float(met[k]) for k in ("accuracy", "precision", "recall", "f1_macro")}},
              open(ck, "w"), indent=2)
    print(f"{tag}: best_ep={best_ep} F1={met['f1_macro']*100:.2f} -> {ck}")
print("A/B done")

dynamo ok


NameError: name '_hf_train' is not defined

In [1]:
import os
import random

import numpy as np
import torch
from tqdm.auto import tqdm

# ---- reproducibility --------------------------------------------------------
SEED = 0


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

# ---- geometry -----------------------------------------------------------
IMG_SIZE = 128
STRIDE = 16                                # single tap point: 128 / 16 = 8x8 feature map
GRID_SIZE = IMG_SIZE // STRIDE             # 8
ASPECT_RATIOS = (1.0, 1.8)                 # 1:1 (fist/palm) + elongated (pointing)
NUM_ANCHORS_PER_CELL = len(ASPECT_RATIOS)  # 2
ANCHOR_BASE_PX = 48                        # base anchor size in pixels on the 128px frame
NUM_TOTAL_ANCHORS = GRID_SIZE * GRID_SIZE * NUM_ANCHORS_PER_CELL  # 8*8*2 = 128

# ---- backbone -------------------------------------------------------------
CUT_INDEX = 14                             # features[:14] -> output stride 16, 8x8
TEACHER_WIDTH_MULT = 1.0                   # torchvision only ships an ImageNet checkpoint here
STUDENT_WIDTH_MULT = 0.3                   # the actual deployable size -- back at the original
                                            # ESP32-class budget, trained from scratch but guided
                                            # by the teacher's soft outputs instead of ground
                                            # truth alone. See model.param_summary() below (run
                                            # for both models) for exact counts.
WIDTH_MULT = STUDENT_WIDTH_MULT            # kept for compatibility with code shared from the
                                            # base/pretrained notebooks that defaults to a single
                                            # global WIDTH_MULT (e.g. train_model's docstrings) --
                                            # both teacher and student are built explicitly with
                                            # their own width below, this is not read for either

# ---- classes ----------------------------------------------------------------
CLASS_NAMES = ["fist", "palm", "like", "ok", "peace", "point"]
NUM_CLASSES = len(CLASS_NAMES)             # background is implicit class 0 in the head output

# ---- box encoding / matching -----------------------------------------------
VARIANCES = (0.1, 0.1, 0.2, 0.2)           # standard SSD-style scale factors for regression targets
POS_IOU_THRESH = 0.5
NEG_IOU_THRESH = 0.4
NEG_POS_RATIO = 3                          # hard-negative-mining ratio

# ---- dataset ----------------------------------------------------------------
HF_DATASET_ID = "cj-mills/hagrid-sample-30k-384p"
HAGRID_TO_PROJECT = {"fist": "fist", "palm": "palm", "like": "like",
                      "ok": "ok", "peace": "peace", "one": "point"}
WANTED_HAGRID_CLASSES = set(HAGRID_TO_PROJECT.keys())
MAX_IMAGES_PER_CLASS = 800                 # per-class cap before MediaPipe box extraction

# ---- training ----------------------------------------------------------------
BATCH_SIZE = 64
TEACHER_EPOCHS = 60   # matches TinyGestureSSD_pretrained.ipynb (Youcef kept it at 60,
                                   # citing diminishing returns given the pretrained head start)                         # stage A: train the teacher to convergence, one
                                             # clean run, same reasoning as the pretrained notebook
STUDENT_EPOCHS = 120                        # stage B: distill into the student. Raise either
                                             # budget if its val_f1 is still climbing when the run
                                             # ends rather than lowering it if it plateaus early --
                                             # the checkpoint saved is always the single best
                                             # epoch, so a longer run never costs you the result.
LR = 2e-3
WEIGHT_DECAY = 1e-4
SELECTION_METRIC = "f1_macro"              # which val metric selects the saved checkpoint

# ---- distillation -----------------------------------------------------------
KD_TEMPERATURE = 2.0                       # softmax temperature for the soft classification
                                            # target -- standard Hinton et al. default, softens
                                            # the teacher's distribution so the student learns
                                            # from relative class confidences, not just the argmax
KD_ALPHA = 0.5                             # weight on the soft classification (KL) term
KD_BETA = 0.5                              # weight on the soft localization (SmoothL1) term

# ---- mediapipe (pseudo-label source for ground-truth boxes) -----------------
MEDIAPIPE_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/latest/hand_landmarker.task"
)

# ---- paths ----------------------------------------------------------------
PROJECT_ROOT = os.getcwd()                 # run this notebook from your project root
MODELS_DIR = os.path.join(PROJECT_ROOT, "models_kd")           # checkpoints/ONNX/plots for THIS
                                                                # experiment only -- kept separate
                                                                # from the base and pretrained
                                                                # notebooks so runs don't collide
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")            # shared across all three notebooks
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)
MEDIAPIPE_MODEL_PATH = os.path.join(DATASET_DIR, "hand_landmarker.task")  # shared asset, not
                                                                            # experiment-specific
DATASET_CACHE_DIR = os.path.join(DATASET_DIR, "hagrid_pseudo_labeled")
CLASSES_FILE = os.path.join(DATASET_DIR, "classes.txt")
_LEGACY_DATASET_CACHE_DIR = os.path.join(PROJECT_ROOT, "models", "hagrid_pseudo_labeled")  # pre-dataset/ layout


# ---- device ----------------------------------------------------------------
def get_device() -> torch.device:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True   # fixed 128x128 input shape -> let cuDNN pick fast kernels
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: no GPU detected -- running on CPU. Training will be much slower.")
    return device


device = get_device()
print(f"grid: {GRID_SIZE}x{GRID_SIZE}  anchors/cell: {NUM_ANCHORS_PER_CELL}  total anchors: {NUM_TOTAL_ANCHORS}")
print(f"classes: {CLASS_NAMES}  (+ background)")

GPU: NVIDIA GeForce RTX 2050
grid: 8x8  anchors/cell: 2  total anchors: 128
classes: ['fist', 'palm', 'like', 'ok', 'peace', 'point']  (+ background)


In [2]:
def generate_anchors(img_size, stride, base_size_px, aspect_ratios):
    """
    Vectorized anchor generation.
    Returns center-form anchors (cx, cy, w, h), normalized to [0, 1], shape (G*G*R, 4).
    Order: row-major over grid cells (row-major = row changes slowest), then aspect
    ratio within each cell (fastest-changing) -- must match the head's channel layout.
    """
    grid_size = img_size // stride
    base = base_size_px / img_size

    rows = torch.arange(grid_size, dtype=torch.float32)
    cols = torch.arange(grid_size, dtype=torch.float32)
    grid_r, grid_c = torch.meshgrid(rows, cols, indexing="ij")     # (G,G), row-major

    cy = ((grid_r + 0.5) * stride / img_size).reshape(-1)          # (G*G,)
    cx = ((grid_c + 0.5) * stride / img_size).reshape(-1)          # (G*G,)

    ratios = torch.tensor(aspect_ratios, dtype=torch.float32)      # (R,)
    R = ratios.shape[0]

    # repeat each cell's center R times contiguously -> ratio is the fast axis
    cx = cx.unsqueeze(1).expand(-1, R).reshape(-1)                 # (G*G*R,)
    cy = cy.unsqueeze(1).expand(-1, R).reshape(-1)

    w = (base * ratios.sqrt()).unsqueeze(0).expand(grid_size * grid_size, -1).reshape(-1)
    h = (base / ratios.sqrt()).unsqueeze(0).expand(grid_size * grid_size, -1).reshape(-1)

    return torch.stack([cx, cy, w, h], dim=-1)


def _generate_anchors_loop_reference(img_size, stride, base_size_px, aspect_ratios):
    """Slow, obviously-correct loop version -- used only to unit-test the vectorized one above."""
    grid_size = img_size // stride
    base = base_size_px / img_size
    out = []
    for row in range(grid_size):
        for col in range(grid_size):
            cx = (col + 0.5) * stride / img_size
            cy = (row + 0.5) * stride / img_size
            for ratio in aspect_ratios:
                w = base * (ratio ** 0.5)
                h = base / (ratio ** 0.5)
                out.append([cx, cy, w, h])
    return torch.tensor(out, dtype=torch.float32)


def verify_anchors(anchors_center, img_size, stride, base_size_px, aspect_ratios):
    """Asserts the vectorized anchors match the loop reference exactly. Call once at startup."""
    ref = _generate_anchors_loop_reference(img_size, stride, base_size_px, aspect_ratios)
    assert torch.allclose(anchors_center, ref), "vectorized anchors don't match the loop reference!"
    print("anchor generation verified against loop reference")


def center_to_corner(boxes):
    """(cx, cy, w, h) -> (x1, y1, x2, y2)"""
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)


def corner_to_center(boxes):
    """(x1, y1, x2, y2) -> (cx, cy, w, h)"""
    x1, y1, x2, y2 = boxes.unbind(-1)
    return torch.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1], dim=-1)


def box_iou(boxes1, boxes2):
    """Vectorized pairwise IoU. boxes1: (N,4), boxes2: (M,4), corner-form -> (N,M)."""
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=1e-8)


def box_iou_batch(gt_boxes_corner, anchors_corner):
    """
    Batched one-box-per-image IoU against the full anchor set -- no Python loop over the batch.
    gt_boxes_corner: (B,4), anchors_corner: (A,4) -> (B,A)
    """
    gt = gt_boxes_corner.unsqueeze(1)          # (B,1,4)
    an = anchors_corner.unsqueeze(0)           # (1,A,4)
    lt = torch.max(gt[..., :2], an[..., :2])
    rb = torch.min(gt[..., 2:], an[..., 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    area_gt = (gt[..., 2] - gt[..., 0]) * (gt[..., 3] - gt[..., 1])
    area_an = (an[..., 2] - an[..., 0]) * (an[..., 3] - an[..., 1])
    union = area_gt + area_an - inter
    return inter / union.clamp(min=1e-8)

In [3]:
anchors_center = generate_anchors(IMG_SIZE, STRIDE, ANCHOR_BASE_PX, ASPECT_RATIOS)
verify_anchors(anchors_center, IMG_SIZE, STRIDE, ANCHOR_BASE_PX, ASPECT_RATIOS)
anchors_corner = center_to_corner(anchors_center)
anchors_center_d = anchors_center.to(device)
anchors_corner_d = anchors_corner.to(device)
print(f"anchors: {tuple(anchors_center.shape)}")

anchor generation verified against loop reference
anchors: (128, 4)


In [4]:
import torch.nn as nn
import torchvision


def build_truncated_backbone(width_mult: float = WIDTH_MULT) -> nn.Module:
    if width_mult == 1.0:
        full = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1)
        print("backbone: ImageNet-pretrained MobileNetV2 (width_mult=1.0)")
    else:
        full = torchvision.models.mobilenet_v2(weights=None, width_mult=width_mult)
        print(f"backbone: from-scratch MobileNetV2 (width_mult={width_mult}) -- "
              f"no pretrained checkpoint exists at this width")
    return nn.Sequential(*list(full.features.children())[:CUT_INDEX])


def probe_backbone_blocks(width_mult: float = 1.0, img_size: int = IMG_SIZE):
    """Prints each MobileNetV2 block's output shape -- useful for picking CUT_INDEX."""
    full = torchvision.models.mobilenet_v2(weights=None, width_mult=width_mult)
    x = torch.zeros(1, 3, img_size, img_size)
    print(f"--- block-by-block probe (width_mult={width_mult}) ---")
    with torch.no_grad():
        for i, block in enumerate(full.features):
            x = block(x)
            print(f"{i:2d}  {type(block).__name__:20s}  {tuple(x.shape)}")

In [5]:
class SeparableHead(nn.Module):
    """SSDLite-style prediction head: depthwise 3x3 (+BN+ReLU6) -> pointwise 1x1."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1,
                                    groups=in_channels, bias=False)
        self.bn = nn.BatchNorm2d(in_channels)
        self.act = nn.ReLU6(inplace=True)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)

    def forward(self, x):
        x = self.act(self.bn(self.depthwise(x)))
        return self.pointwise(x)


def reshape_head_output(x, num_anchors_per_cell: int, out_per_anchor: int):
    """(B, A_cell*K, H, W) -> (B, H*W*A_cell, K), row-major cells then ratio-within-cell."""
    B, _, H, W = x.shape
    x = x.view(B, num_anchors_per_cell, out_per_anchor, H, W)
    x = x.permute(0, 3, 4, 1, 2).contiguous()          # (B, H, W, A_cell, K)
    return x.view(B, H * W * num_anchors_per_cell, out_per_anchor)


class SSDLiteGestureDetector(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, width_mult: float = WIDTH_MULT,
                 num_anchors_per_cell: int = NUM_ANCHORS_PER_CELL, img_size: int = IMG_SIZE):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors_per_cell = num_anchors_per_cell
        self.width_mult = width_mult               # stored on the instance, not read from the
                                                     # global at print time -- matters once there's
                                                     # more than one model instance at different
                                                     # widths, like the teacher/student pair here

        self.backbone = build_truncated_backbone(width_mult)

        # channel count is *inferred*, never hard-coded, so width_mult stays a single knob
        with torch.no_grad():
            feat = self.backbone(torch.zeros(1, 3, img_size, img_size))
        in_channels, grid = feat.shape[1], feat.shape[2]
        self.in_channels, self.grid_size = in_channels, grid

        self.cls_head = SeparableHead(in_channels, num_anchors_per_cell * (num_classes + 1))
        self.loc_head = SeparableHead(in_channels, num_anchors_per_cell * 4)

    def forward(self, x):
        feat = self.backbone(x)
        cls_out = reshape_head_output(self.cls_head(feat), self.num_anchors_per_cell, self.num_classes + 1)
        loc_out = reshape_head_output(self.loc_head(feat), self.num_anchors_per_cell, 4)
        return cls_out, loc_out                         # (B, num_anchors, C+1), (B, num_anchors, 4)

    def param_summary(self):
        def n(m):
            return sum(p.numel() for p in m.parameters())

        bb, cls_h, loc_h = n(self.backbone), n(self.cls_head), n(self.loc_head)
        total = bb + cls_h + loc_h
        print(f"{'backbone (mbv2 features[:14])':35s} {bb:>8,}")
        print(f"{'classification head':35s} {cls_h:>8,}")
        print(f"{'localization head':35s} {loc_h:>8,}")
        print("-" * 44)
        print(f"{'TOTAL':35s} {total:>8,}   (width_mult={self.width_mult}; original on-device "
              f"budget was ~100,000 @ width_mult=0.35)")
        return total

In [6]:
def encode_boxes(gt_center, anchor_center, variances=VARIANCES):
    """Ground-truth center-form boxes -> SSD-style regression targets relative to their anchor."""
    g_cx, g_cy, g_w, g_h = gt_center.unbind(-1)
    a_cx, a_cy, a_w, a_h = anchor_center.unbind(-1)
    v1, v2, v3, v4 = variances
    t_cx = (g_cx - a_cx) / a_w / v1
    t_cy = (g_cy - a_cy) / a_h / v2
    t_w = torch.log(g_w / a_w) / v3
    t_h = torch.log(g_h / a_h) / v4
    return torch.stack([t_cx, t_cy, t_w, t_h], dim=-1)


def decode_boxes(pred_offsets, anchor_center, variances=VARIANCES):
    """Inverse of encode_boxes -- predicted offsets + anchor -> absolute center-form box."""
    t_cx, t_cy, t_w, t_h = pred_offsets.unbind(-1)
    a_cx, a_cy, a_w, a_h = anchor_center.unbind(-1)
    v1, v2, v3, v4 = variances
    p_cx = t_cx * v1 * a_w + a_cx
    p_cy = t_cy * v2 * a_h + a_cy
    p_w = torch.exp(t_w * v3) * a_w
    p_h = torch.exp(t_h * v4) * a_h
    return torch.stack([p_cx, p_cy, p_w, p_h], dim=-1)


def match_anchors_to_targets(gt_boxes_corner, gt_labels, anchors_center, anchors_corner,
                              pos_iou_thresh=POS_IOU_THRESH, neg_iou_thresh=NEG_IOU_THRESH):
    """
    Vectorized single-object-per-image anchor matching.
      gt_boxes_corner: (B,4) normalized (x1,y1,x2,y2)
      gt_labels:       (B,)  0-indexed foreground class id
    Returns:
      cls_targets: (B,A) long  -- 0=background, 1..num_classes=foreground, -1=ignore
      loc_targets: (B,A,4)     -- encoded regression targets (only valid where pos_mask is True)
      pos_mask:    (B,A) bool
      ignore_mask: (B,A) bool  -- 0.4 <= iou < 0.5, excluded from the loss entirely
    """
    B, A = gt_boxes_corner.shape[0], anchors_center.shape[0]

    iou = box_iou_batch(gt_boxes_corner, anchors_corner)            # (B,A)

    pos_mask = iou >= pos_iou_thresh
    forced_idx = iou.argmax(dim=1, keepdim=True)                    # guarantee >=1 positive/image
    pos_mask.scatter_(1, forced_idx, True)

    neg_mask = (iou < neg_iou_thresh) & (~pos_mask)
    ignore_mask = (~pos_mask) & (~neg_mask)

    cls_targets = torch.zeros(B, A, dtype=torch.long, device=gt_boxes_corner.device)
    cls_targets[pos_mask] = (gt_labels.unsqueeze(1).expand(B, A) + 1)[pos_mask]
    cls_targets[ignore_mask] = -1

    gt_center = corner_to_center(gt_boxes_corner)                   # (B,4)
    gt_center_exp = gt_center.unsqueeze(1).expand(B, A, 4)
    anchor_center_exp = anchors_center.unsqueeze(0).expand(B, A, 4)
    loc_targets = encode_boxes(gt_center_exp, anchor_center_exp)    # (B,A,4)

    return cls_targets, loc_targets, pos_mask, ignore_mask

In [7]:
import torch.nn.functional as F


class MultiBoxLoss(nn.Module):
    def __init__(self, neg_pos_ratio: int = NEG_POS_RATIO):
        super().__init__()
        self.neg_pos_ratio = neg_pos_ratio

    def forward(self, cls_preds, loc_preds, cls_targets, loc_targets, pos_mask, ignore_mask):
        B, A, C = cls_preds.shape

        # ---- localization: positives only ----
        loc_loss = F.smooth_l1_loss(loc_preds[pos_mask], loc_targets[pos_mask], reduction="sum")

        # ---- classification with 3:1 hard negative mining ----
        safe_targets = cls_targets.clamp(min=0)   # -1 (ignore) -> 0 just so CE doesn't error; excluded below
        loss_per_anchor = F.cross_entropy(
            cls_preds.view(-1, C), safe_targets.view(-1), reduction="none"
        ).view(B, A)

        loss_for_ranking = loss_per_anchor.clone()
        loss_for_ranking[pos_mask] = float("-inf")     # never select positives as "negatives"
        loss_for_ranking[ignore_mask] = float("-inf")  # never select ignored anchors either

        num_pos = pos_mask.sum(dim=1, keepdim=True)                          # (B,1)
        num_neg_avail = (~pos_mask & ~ignore_mask).sum(dim=1, keepdim=True)
        num_neg = torch.clamp(self.neg_pos_ratio * num_pos, max=num_neg_avail)

        _, loss_idx = loss_for_ranking.sort(dim=1, descending=True)
        _, rank = loss_idx.sort(dim=1)               # rank[b,a] = position of anchor a in the sorted order
        hard_neg_mask = rank < num_neg

        selected = pos_mask | hard_neg_mask
        cls_loss = loss_per_anchor[selected].sum()

        num_pos_total = num_pos.sum().clamp(min=1).float()
        loc_loss = loc_loss / num_pos_total
        cls_loss = cls_loss / num_pos_total
        return loc_loss + cls_loss, loc_loss, cls_loss

In [8]:
import shutil
import time

from datasets import load_dataset, load_from_disk
from huggingface_hub import login

# ---------------------------------------------------------------------------
# mediapipe import guard
# ---------------------------------------------------------------------------
# A NameError like "name 'audio_classifier' is not defined" raised from inside
# mediapipe's own __init__.py (not from your code) means the *installed package*
# is broken -- it is not something this notebook's code can work around. The two
# usual causes:
#   1. A local file/folder named `mediapipe.py` or `mediapipe/` somewhere on
#      sys.path (often the project root, i.e. os.getcwd()) is shadowing the real
#      installed package.
#   2. A corrupted/partial pip install (very common on Windows if a previous
#      mediapipe process still had files locked during install/upgrade).
# This cell checks for (1) automatically and gives an actionable message for (2).
import importlib.util as _ilu
import os as _os

_shadow = _os.path.join(_os.getcwd(), "mediapipe.py")
if _os.path.exists(_shadow):
    raise RuntimeError(
        f"Found {_shadow} -- this file is shadowing the real 'mediapipe' package "
        "and must be renamed or deleted before mediapipe can be imported."
    )

try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
except NameError as e:
    raise RuntimeError(
        "mediapipe failed to import cleanly (this is a broken local install, not a "
        "notebook bug). Fix it in a terminal (not this notebook) with:\n"
        "    pip uninstall -y mediapipe\n"
        "    pip cache purge\n"
        "    pip install --no-cache-dir mediapipe==0.10.14\n"
        "then restart the kernel and re-run this cell. If it still fails, check for a "
        "stray mediapipe.py/mediapipe/ file on sys.path (see check above) and that no "
        "other package (e.g. an old 'mp' alias) is colliding with it."
    ) from e


# ---------------------------------------------------------------------------
# HaGRID download + per-class capping
# ---------------------------------------------------------------------------

def _load_raw_hagrid():
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("logged in to Hugging Face Hub via HF_TOKEN env var")

    print(f"downloading / loading cached copy of {HF_DATASET_ID} ...")
    raw_ds = load_dataset(HF_DATASET_ID)

    hagrid_label_names = [name.replace("train_val_", "")
                           for name in raw_ds["train"].features["label"].names]
    print("HaGRID classes in this sample:", hagrid_label_names)
    return raw_ds, hagrid_label_names


def _cap_per_class(raw_ds, hagrid_label_names):
    all_labels = raw_ds["train"]["label"]
    print(f"read {len(all_labels)} labels")

    rng = random.Random(SEED)
    by_class_idx = {c: [] for c in WANTED_HAGRID_CLASSES}
    order = list(range(len(all_labels)))
    rng.shuffle(order)

    pbar = tqdm(order, desc="capping per class")
    for i in pbar:
        cname = hagrid_label_names[all_labels[i]]
        if cname in WANTED_HAGRID_CLASSES and len(by_class_idx[cname]) < MAX_IMAGES_PER_CLASS:
            by_class_idx[cname].append(i)
        if all(len(v) >= MAX_IMAGES_PER_CLASS for v in by_class_idx.values()):
            pbar.set_postfix(status="cap reached, stopping early")
            break

    keep_indices = sorted(idx for idxs in by_class_idx.values() for idx in idxs)
    capped = raw_ds["train"].select(keep_indices)

    print("materializing the capped subset into its own compact file...")
    capped = capped.flatten_indices()

    print("per-class counts after capping:", {c: len(v) for c, v in by_class_idx.items()})
    print(f"total images going into box extraction: {len(capped)}")
    return capped


# ---------------------------------------------------------------------------
# MediaPipe Hand Landmarker pseudo-labeling
# ---------------------------------------------------------------------------

def _detect_hand_bbox(landmarker, pil_image, pad_frac=0.10):
    """
    Runs the real Hand Landmarker (not a color heuristic) on one PIL image.
    Returns (x1, y1, x2, y2) normalized to [0,1], or None if no hand found.

    If multiple hands are detected, picks the one with the larger bounding box -- the
    gesture-performing hand is usually the more prominent one. MediaPipe has no way to
    know which hand matches this row's *label*, only that a hand is present, so this is
    still a heuristic pick, just a much more reliable one than color thresholding.
    """
    rgb = np.array(pil_image.convert("RGB"))
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    if not result.hand_landmarks:
        return None

    candidates = []
    for hand in result.hand_landmarks:
        xs = [lm.x for lm in hand]
        ys = [lm.y for lm in hand]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        candidates.append((x1, y1, x2, y2, (x2 - x1) * (y2 - y1)))

    x1, y1, x2, y2, _ = max(candidates, key=lambda c: c[4])   # largest box wins

    pw, ph = (x2 - x1) * pad_frac, (y2 - y1) * pad_frac
    x1, x2 = max(0.0, x1 - pw), min(1.0, x2 + pw)
    y1, y2 = max(0.0, y1 - ph), min(1.0, y2 + ph)
    return (x1, y1, x2, y2)


def _extract_boxes_with_mediapipe(capped, hagrid_label_names):
    if not os.path.exists(MEDIAPIPE_MODEL_PATH):
        import urllib.request
        print(f"downloading hand landmarker model to {MEDIAPIPE_MODEL_PATH} ...")
        urllib.request.urlretrieve(MEDIAPIPE_MODEL_URL, MEDIAPIPE_MODEL_PATH)
        print(f"  done, {os.path.getsize(MEDIAPIPE_MODEL_PATH) / 1e6:.1f} MB")
    else:
        print(f"model already present at {MEDIAPIPE_MODEL_PATH}")

    base_options = mp_python.BaseOptions(model_asset_path=MEDIAPIPE_MODEL_PATH)
    landmarker_options = mp_vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=2,
        running_mode=mp_vision.RunningMode.IMAGE,
    )
    landmarker = mp_vision.HandLandmarker.create_from_options(landmarker_options)

    # pull one row at a time instead of `capped["image"]`, which decodes every image in the
    # split into a Python list up front -- for a few thousand images that's a multi-GB spike
    # held in RAM simultaneously, on top of PyTorch/MediaPipe/everything else already running.
    # on an 8GB machine that's enough to push the system into swap, which looks exactly like a
    # hang: CPU usage drops, nothing crashes, nothing progresses. streaming keeps peak memory
    # roughly constant regardless of dataset size.
    n_images = len(capped)
    final_boxes, keep_mask = [], []
    pbar = tqdm(range(n_images), desc="MediaPipe hand detection", unit="img")
    t0 = time.time()
    for i in pbar:
        img = capped[i]["image"]
        box = _detect_hand_bbox(landmarker, img)
        keep_mask.append(box is not None)
        final_boxes.append(box)
        n_done = len(final_boxes)
        pbar.set_postfix(hit_rate=f"{sum(keep_mask) / n_done:.0%}")
    elapsed = time.time() - t0
    print(f"  done in {elapsed:.1f}s  ({elapsed / max(n_images, 1) * 1000:.0f} ms/image)")

    n_found = sum(keep_mask)
    print(f"hand detected in {n_found}/{n_images} images ({n_found / n_images:.1%})")

    detected_indices = [i for i, keep in enumerate(keep_mask) if keep]
    final_ds = capped.select(detected_indices)
    final_ds = final_ds.add_column("bbox", [final_boxes[i] for i in detected_indices])

    class_counts = {c: 0 for c in CLASS_NAMES}
    for lbl in final_ds["label"]:
        class_counts[HAGRID_TO_PROJECT[hagrid_label_names[lbl]]] += 1
    print("final class balance:", class_counts)
    return final_ds


# ---------------------------------------------------------------------------
# Training-time augmentation (box-coordinate-aware)
# ---------------------------------------------------------------------------

_COLOR_JITTER = torchvision.transforms.ColorJitter(brightness=0.25, contrast=0.25,
                                                      saturation=0.2, hue=0.02)


def _hflip(img, box_norm):
    """50% chance horizontal flip. Safe for this class list -- none of CLASS_NAMES are
    direction-sensitive (`point` is one class regardless of which way the finger points)."""
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        x1, y1, x2, y2 = box_norm
        box_norm = np.array([1.0 - x2, y1, 1.0 - x1, y2], dtype=np.float32)
    return img, box_norm


def _scale_jitter(img, box_norm, scale_range=(0.85, 1.15)):
    """
    Zooms in/out around the image center by a random factor and returns an image of the
    same size. A point at normalized position p maps to 0.5 + (p-0.5)*scale regardless of
    zoom direction -- only the PIL operation differs (crop+resize to zoom in,
    shrink+pad to zoom out). img must already be square (IMG_SIZE x IMG_SIZE).
    """
    scale = random.uniform(*scale_range)
    S = img.size[0]

    if scale >= 1.0:
        crop = S / scale
        left = top = (S - crop) / 2
        img = img.crop((left, top, left + crop, top + crop)).resize((S, S), Image.BILINEAR)
    else:
        new_s = max(1, int(S * scale))
        img_small = img.resize((new_s, new_s), Image.BILINEAR)
        pad_color = tuple(np.array(img).reshape(-1, 3).mean(axis=0).astype(int).tolist())
        canvas = Image.new("RGB", (S, S), pad_color)
        canvas.paste(img_small, ((S - new_s) // 2, (S - new_s) // 2))
        img = canvas

    x1, y1, x2, y2 = box_norm
    x1, x2 = 0.5 + (x1 - 0.5) * scale, 0.5 + (x2 - 0.5) * scale
    y1, y2 = 0.5 + (y1 - 0.5) * scale, 0.5 + (y2 - 0.5) * scale
    return img, np.clip([x1, y1, x2, y2], 0.0, 1.0).astype(np.float32)


def augment_sample(img, box_norm):
    """img: PIL Image already resized to IMG_SIZE x IMG_SIZE. box_norm: [x1,y1,x2,y2] in [0,1]."""
    img, box_norm = _hflip(img, box_norm)
    img, box_norm = _scale_jitter(img, box_norm)
    img = _COLOR_JITTER(img)   # geometry-independent -- always last
    return img, box_norm


# ---------------------------------------------------------------------------
# PyTorch Dataset / DataLoaders
# ---------------------------------------------------------------------------

class HaGRIDGestureDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, hagrid_label_names, img_size: int = IMG_SIZE, augment: bool = False):
        self.ds = hf_split
        self.hagrid_label_names = hagrid_label_names
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        img = row["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        box_norm = np.array(row["bbox"], dtype=np.float32)
        box_norm = np.clip(box_norm, 0.0, 1.0)

        if self.augment:
            img, box_norm = augment_sample(img, box_norm)

        box_t = torch.from_numpy(box_norm).clamp(0.0, 1.0)

        label_name = HAGRID_TO_PROJECT[self.hagrid_label_names[row["label"]]]
        label_t = torch.tensor(CLASS_NAMES.index(label_name), dtype=torch.long)

        image_t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        return image_t, box_t, label_t


def prepare_dataset(force_rebuild: bool = False, augment_train: bool = True):
    """
    Returns (train_loader, val_loader, hagrid_label_names).

    Caches the MediaPipe-pseudo-labeled dataset to DATASET_CACHE_DIR (under the top-level
    dataset/ folder) so the (slow) MediaPipe pass only ever runs once per
    MAX_IMAGES_PER_CLASS/SEED combination. Pass force_rebuild=True to ignore an existing cache
    and recompute it.

    If a dataset from an older run of this notebook is still sitting at the legacy
    models/hagrid_pseudo_labeled location, it gets moved into dataset/ instead of recomputed.

    augment_train=True applies random flip/scale/color augmentation to the train split only.
    """
    raw_ds, hagrid_label_names = _load_raw_hagrid()   # cheap; needed either way for label names

    if (not os.path.isdir(DATASET_CACHE_DIR)) and os.path.isdir(_LEGACY_DATASET_CACHE_DIR):
        print(f"found a dataset cached at the old location ({_LEGACY_DATASET_CACHE_DIR}) -- "
              f"moving it to {DATASET_CACHE_DIR} instead of recomputing it")
        shutil.move(_LEGACY_DATASET_CACHE_DIR, DATASET_CACHE_DIR)

    if os.path.isdir(DATASET_CACHE_DIR) and not force_rebuild:
        print(f"found cached pseudo-labeled dataset at {DATASET_CACHE_DIR}, skipping MediaPipe pass")
        final_ds = load_from_disk(DATASET_CACHE_DIR)
    else:
        capped = _cap_per_class(raw_ds, hagrid_label_names)
        final_ds = _extract_boxes_with_mediapipe(capped, hagrid_label_names)
        print(f"caching pseudo-labeled dataset to {DATASET_CACHE_DIR} for future runs...")
        final_ds.save_to_disk(DATASET_CACHE_DIR)

    if not os.path.exists(CLASSES_FILE):
        with open(CLASSES_FILE, "w") as f:
            f.write("\n".join(CLASS_NAMES) + "\n")
        print(f"wrote class list to {CLASSES_FILE}")

    final_ds = final_ds.shuffle(seed=SEED)
    n_val = max(1, int(0.1 * len(final_ds)))
    hf_val = final_ds.select(range(n_val))
    hf_train = final_ds.select(range(n_val, len(final_ds)))
    print(f"train: {len(hf_train)}   val: {len(hf_val)}   (augment_train={augment_train})")

    train_ds = HaGRIDGestureDataset(hf_train, hagrid_label_names, augment=augment_train)
    val_ds = HaGRIDGestureDataset(hf_val, hagrid_label_names, augment=False)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                                 num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                               num_workers=0, pin_memory=torch.cuda.is_available())

    imgs, boxes, labels = next(iter(train_loader))
    print("batch:", tuple(imgs.shape), tuple(boxes.shape), tuple(labels.shape))

    return train_loader, val_loader, hagrid_label_names

In [9]:
from PIL import Image

train_loader, val_loader, hagrid_label_names = prepare_dataset()

downloading / loading cached copy of cj-mills/hagrid-sample-30k-384p ...


Using the latest cached version of the dataset since cj-mills/hagrid-sample-30k-384p couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\xared\.cache\huggingface\datasets\cj-mills___hagrid-sample-30k-384p\default\0.0.0\40b00da827c57b98efb5bc73f5abf49307c713d4 (last modified on Thu Aug 13 02:48:29 2026).


HaGRID classes in this sample: ['call', 'dislike', 'fist', 'four', 'like', 'mute', 'ok', 'one', 'palm', 'peace', 'peace_inverted', 'rock', 'stop', 'stop_inverted', 'three', 'three2', 'two_up', 'two_up_inverted']
found cached pseudo-labeled dataset at c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\dataset\hagrid_pseudo_labeled, skipping MediaPipe pass
train: 3906   val: 433   (augment_train=True)
batch: (64, 3, 128, 128) (64, 4) (64,)


In [10]:
def _box_wh_px(hf_split):
    boxes = np.array(hf_split["bbox"], dtype=np.float32)   # (N,4) x1,y1,x2,y2 normalized
    w = (boxes[:, 2] - boxes[:, 0]) * IMG_SIZE
    h = (boxes[:, 3] - boxes[:, 1]) * IMG_SIZE
    return w, h

w_train, h_train = _box_wh_px(train_loader.dataset.ds)
w_val, h_val = _box_wh_px(val_loader.dataset.ds)
w_all = np.concatenate([w_train, w_val])
h_all = np.concatenate([h_train, h_val])
ratio_all = w_all / h_all
scale_all = np.sqrt(w_all * h_all)

print(f"box width  (px): min={w_all.min():.1f}  median={np.median(w_all):.1f}  max={w_all.max():.1f}")
print(f"box height (px): min={h_all.min():.1f}  median={np.median(h_all):.1f}  max={h_all.max():.1f}")
print(f"box w/h ratio:    min={ratio_all.min():.2f}  median={np.median(ratio_all):.2f}  max={ratio_all.max():.2f}")

fitted_base_px = float(np.median(scale_all))
fitted_ratio_lo = float(np.percentile(ratio_all, 25))
fitted_ratio_hi = float(np.percentile(ratio_all, 75))

print(f"\nold  ANCHOR_BASE_PX={ANCHOR_BASE_PX}   ASPECT_RATIOS={ASPECT_RATIOS}")
print(f"new  ANCHOR_BASE_PX={fitted_base_px:.1f}   ASPECT_RATIOS=({fitted_ratio_lo:.2f}, {fitted_ratio_hi:.2f})")

ANCHOR_BASE_PX = fitted_base_px
ASPECT_RATIOS = (fitted_ratio_lo, fitted_ratio_hi)

anchors_center = generate_anchors(IMG_SIZE, STRIDE, ANCHOR_BASE_PX, ASPECT_RATIOS)
anchors_corner = center_to_corner(anchors_center)
anchors_center_d = anchors_center.to(device)
anchors_corner_d = anchors_corner.to(device)
print(f"\nanchors regenerated: {tuple(anchors_center.shape)}  (count unchanged, geometry refit)")

box width  (px): min=1.9  median=15.1  max=74.1
box height (px): min=4.9  median=19.8  max=106.4
box w/h ratio:    min=0.15  median=0.84  max=3.47

old  ANCHOR_BASE_PX=48   ASPECT_RATIOS=(1.0, 1.8)
new  ANCHOR_BASE_PX=17.1   ASPECT_RATIOS=(0.57, 1.08)

anchors regenerated: (128, 4)  (count unchanged, geometry refit)


In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


@torch.no_grad()
def predict_batch_labels(model, imgs, anchors_center, conf_thresh: float = 0.5):
    """
    Returns pred_labels: (B,) long tensor, project class id, or -1 if nothing cleared
    conf_thresh. Used for fast per-epoch validation (batched version of the single-image
    detector in Section 9). Works on either teacher or student -- takes any model with the
    same forward signature.
    """
    cls_preds, loc_preds = model(imgs)                       # (B,A,C+1), (B,A,4)
    probs = F.softmax(cls_preds, dim=-1)
    fg_probs = probs[..., 1:]
    best_score_per_anchor, best_cls_per_anchor = fg_probs.max(dim=-1)      # (B,A)

    thresholded = best_score_per_anchor.masked_fill(best_score_per_anchor < conf_thresh, -1.0)
    best_score, best_anchor_idx = thresholded.max(dim=-1)                  # (B,)

    pred_labels = best_cls_per_anchor.gather(1, best_anchor_idx.unsqueeze(1)).squeeze(1)
    pred_labels = torch.where(best_score >= 0, pred_labels, torch.full_like(pred_labels, -1))
    return pred_labels


@torch.no_grad()
def evaluate(model, loader, anchors_center, device, conf_thresh: float = 0.5):
    """Runs the detector over a loader and computes classification-style metrics on the
    gesture label of the single detected hand (a missed detection counts as a wrong
    prediction). Reused as-is for both teacher and student -- it only needs a model with the
    standard forward signature, doesn't care how it was trained."""
    model.eval()
    all_preds, all_gts = [], []
    for imgs, boxes, labels in tqdm(loader, desc="validating", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        preds = predict_batch_labels(model, imgs, anchors_center, conf_thresh)
        all_preds.append(preds.cpu())
        all_gts.append(labels.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_gts = torch.cat(all_gts).numpy()

    accuracy = accuracy_score(all_gts, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_gts, all_preds, labels=list(range(NUM_CLASSES)), average="macro", zero_division=0
    )
    n_missed = int((all_preds == -1).sum())
    return {"accuracy": accuracy, "precision": precision, "recall": recall,
            "f1_macro": f1, "n_missed": n_missed, "n_total": len(all_gts)}


def train_model(model, train_loader, val_loader, anchors_center, anchors_corner, device,
                 epochs: int = TEACHER_EPOCHS, lr: float = LR, weight_decay: float = WEIGHT_DECAY,
                 selection_metric: str = SELECTION_METRIC, checkpoint_name: str = "best_model.pt"):
    """Standard (non-distillation) training loop. `checkpoint_name` lets this be reused for
    the teacher (saved as teacher_best.pt) without colliding with the student's best_model.pt."""
    criterion = MultiBoxLoss().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {
        "total": [], "loc": [], "cls": [],
        "val_accuracy": [], "val_precision": [], "val_recall": [], "val_f1_macro": [],
        "iter_total": [], "iter_loc": [], "iter_cls": [],   # per-iteration, never reset
    }

    best_metric_value = -1.0
    best_state_dict = None
    best_epoch = -1

    epoch_bar = tqdm(range(1, epochs + 1), desc="training", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        running = {"total": 0.0, "loc": 0.0, "cls": 0.0}
        n_batches = 0

        batch_bar = tqdm(train_loader, desc=f"epoch {epoch}/{epochs}", leave=False)
        for imgs, boxes, labels in batch_bar:
            imgs, boxes, labels = imgs.to(device), boxes.to(device), labels.to(device)

            cls_preds, loc_preds = model(imgs)
            cls_t, loc_t, pos_m, ign_m = match_anchors_to_targets(boxes, labels, anchors_center, anchors_corner)
            total_loss, loc_loss, cls_loss = criterion(cls_preds, loc_preds, cls_t, loc_t, pos_m, ign_m)

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            history["iter_total"].append(total_loss.item())
            history["iter_loc"].append(loc_loss.item())
            history["iter_cls"].append(cls_loss.item())

            running["total"] += total_loss.item()
            running["loc"] += loc_loss.item()
            running["cls"] += cls_loss.item()
            n_batches += 1

            batch_bar.set_postfix(total=running["total"] / n_batches,
                                   loc=running["loc"] / n_batches,
                                   cls=running["cls"] / n_batches)

        scheduler.step()
        for k in ("total", "loc", "cls"):
            history[k].append(running[k] / n_batches)

        val_metrics = evaluate(model, val_loader, anchors_center, device)
        for k in ("accuracy", "precision", "recall", "f1_macro"):
            history[f"val_{k}"].append(val_metrics[k])

        is_best = val_metrics[selection_metric] > best_metric_value
        if is_best:
            best_metric_value = val_metrics[selection_metric]
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch

        epoch_bar.set_postfix(total=f"{history['total'][-1]:.3f}",
                               val_f1=f"{val_metrics['f1_macro']:.3f}",
                               best=f"{best_metric_value:.3f}@{best_epoch}")

        print(f"epoch {epoch:2d}/{epochs}  total={history['total'][-1]:.4f}  "
              f"loc={history['loc'][-1]:.4f}  cls={history['cls'][-1]:.4f}  |  "
              f"val_acc={val_metrics['accuracy']:.3f}  val_prec={val_metrics['precision']:.3f}  "
              f"val_rec={val_metrics['recall']:.3f}  val_f1={val_metrics['f1_macro']:.3f}  "
              f"(missed {val_metrics['n_missed']}/{val_metrics['n_total']})"
              f"{'  <-- best so far' if is_best else ''}")

    print(f"\nbest checkpoint: epoch {best_epoch}  {selection_metric}={best_metric_value:.4f}")
    best_model_path = os.path.join(MODELS_DIR, checkpoint_name)
    torch.save(best_state_dict, best_model_path)
    model.load_state_dict(best_state_dict)   # switch the live model to the best checkpoint
    print(f"loaded best checkpoint into the passed-in model and saved it to {best_model_path}")

    return history, best_epoch, best_metric_value

In [12]:
import hashlib, json, os
import numpy as np, torch

REV_DIR = os.path.join(PROJECT_ROOT, "revision_results")
os.makedirs(os.path.join(REV_DIR, "preds"), exist_ok=True)

assert SEED == 0, "split must stay seed 0 to match the paper"
n_train, n_val = len(train_loader.dataset), len(val_loader.dataset)
val_labels = np.array([int(l) for _, _, l in val_loader.dataset])   # fixed order (shuffle=False)
fp = hashlib.md5(val_labels.tobytes()).hexdigest()
print(f"train={n_train}  val={n_val}  val-label fingerprint={fp}")
print("val class counts:", np.bincount(val_labels, minlength=NUM_CLASSES).tolist())
assert (n_train, n_val) == (3906, 433), "split size differs from the paper (3,906 / 433)"

fp_path = os.path.join(REV_DIR, "split_fingerprint.json")
if os.path.exists(fp_path):
    assert json.load(open(fp_path))["val_label_md5"] == fp, "validation split CHANGED vs a previous run!"
else:
    json.dump({"val_label_md5": fp, "n_train": n_train, "n_val": n_val, "seed": SEED}, open(fp_path, "w"), indent=2)
np.save(os.path.join(REV_DIR, "preds", "gts.npy"), val_labels)

train=3906  val=433  val-label fingerprint=d210029e2814357ca28251390535186e
val class counts: [71, 71, 69, 72, 70, 80]


In [13]:
import sys
import os
for m in [m for m in sys.modules if m == "torch._dynamo" or m.startswith("torch._dynamo.")]:
    del sys.modules[m]
import torch._dynamo
print("dynamo ok")
def _strip(fn):
    seen = 0
    while hasattr(fn, "__wrapped__") and seen < 10:
        fn = fn.__wrapped__
        seen += 1
    return torch.no_grad()(fn)
import torch.optim.optimizer as _OO
for _cls in (torch.optim.SGD, torch.optim.AdamW):
    for _m in ("add_param_group", "step", "zero_grad"):
        _raw = _strip(getattr(_cls, _m))
        if getattr(_cls, _m) is not _raw:
            setattr(_cls, _m, _raw)
SEEDS_DIR = os.path.join(PROJECT_ROOT, "models_seeds"); os.makedirs(SEEDS_DIR, exist_ok=True)
print("optimizers stripped; SEEDS_DIR =", SEEDS_DIR)

dynamo ok
optimizers stripped; SEEDS_DIR = c:\Users\xared\Documents\30_PROJECTS\TinyGestureSSD\TinyGestureSSD\gesturessd\models_seeds


In [14]:
import json, torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
fp = json.load(open(os.path.join(REV_DIR, "split_fingerprint.json")))
assert fp["val_label_md5"] == "d210029e2814357ca28251390535186e", "split changed!"
_final = load_from_disk(DATASET_CACHE_DIR).shuffle(seed=SEED)
_nv = max(1, int(0.1 * len(_final)))
_hf_val, _hf_train = _final.select(range(_nv)), _final.select(range(_nv, len(_final)))
TRDS_ON = HaGRIDGestureDataset(_hf_train, hagrid_label_names, augment=True)
TRDS_OFF = HaGRIDGestureDataset(_hf_train, hagrid_label_names, augment=False)
assert (np.array([int(l) for _, _, l in HaGRIDGestureDataset(
    _hf_val, hagrid_label_names, augment=False)]) == np.load(
    os.path.join(REV_DIR, "preds", "gts.npy"))).all(), "val drifted!"
def fresh_loader_for(ds, seed, batch=BATCH_SIZE):
    return DataLoader(ds, batch_size=batch, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))
print(f"paired ready: train={len(_hf_train)} val={len(_hf_val)} (aug toggles ONLY in train)")

paired ready: train=3906 val=433 (aug toggles ONLY in train)


In [15]:
def train_job(kind, width, seed, aug_tag, epochs=120, force_random_init=False):
    global MODELS_DIR
    tag = f"{kind}_{width:.2f}_{aug_tag}_s{seed}".replace(".", "p")
    ckpt = os.path.join(REV_DIR, f"abl_{tag}.json")
    if os.path.exists(ckpt):
        print(tag, "cached, skipping"); return json.load(open(ckpt))
    set_seed(seed)
    ds = TRDS_ON if aug_tag == "aug" else TRDS_OFF
    tl = fresh_loader_for(ds, seed)
    m = SSDLiteGestureDetector(width_mult=width).to(device)
    if force_random_init:
        m.apply(lambda mod: mod.reset_parameters() if isinstance(mod, (nn.Conv2d, nn.Linear, nn.BatchNorm2d)) else None)
        print("forced random init (true from-scratch at width 1.0)")
    _old = MODELS_DIR
    MODELS_DIR = os.path.join(SEEDS_DIR, f"abl_{tag}"); os.makedirs(MODELS_DIR, exist_ok=True)
    try:
        hist, best_ep, best_f1 = train_model(m, tl, val_loader, anchors_center_d, anchors_corner_d,
            device, epochs=epochs, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_name="best_model.pt")
    finally:
        MODELS_DIR = _old
    met = evaluate(m, val_loader, anchors_center_d, device)
    m.eval(); P = []
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            P.append(predict_batch_labels(m, imgs.to(device), anchors_center_d).cpu())
    P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
    np.save(os.path.join(REV_DIR, "preds", f"abl_{tag}.npy"), P)
    rec = {"kind": kind, "width": width, "aug": aug_tag, "seed": seed, "best_epoch": best_ep,
           "force_random_init": force_random_init,
           "params": sum(p.numel() for p in m.parameters()),
           "fp32": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")},
           "n_missed": met["n_missed"]}
    json.dump(rec, open(ckpt, "w"), indent=2)
    print(f"{tag} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} -> {ckpt}")
    return rec

In [16]:
class DistillationLoss(nn.Module):
    def __init__(self, neg_pos_ratio=NEG_POS_RATIO, temperature=KD_TEMPERATURE,
                 alpha=KD_ALPHA, beta=KD_BETA):
        super().__init__()
        self.hard_loss = MultiBoxLoss(neg_pos_ratio=neg_pos_ratio)
        self.temperature = temperature
        self.alpha = alpha
        self.beta = beta

    def forward(self, student_cls, student_loc, teacher_cls, teacher_loc,
                cls_targets, loc_targets, pos_mask, ignore_mask):
        hard_total, hard_loc, hard_cls = self.hard_loss(
            student_cls, student_loc, cls_targets, loc_targets, pos_mask, ignore_mask)
        T = self.temperature
        student_log_soft = F.log_softmax(student_cls / T, dim=-1)
        teacher_soft = F.softmax(teacher_cls / T, dim=-1).detach()
        soft_cls = F.kl_div(student_log_soft, teacher_soft, reduction="batchmean") * (T ** 2)
        num_pos = pos_mask.sum().clamp(min=1).float()
        soft_loc = F.smooth_l1_loss(student_loc[pos_mask], teacher_loc[pos_mask].detach(),
                                     reduction="sum") / num_pos
        total = hard_total + self.alpha * soft_cls + self.beta * soft_loc
        return total, hard_loc, hard_cls, soft_cls, soft_loc

def train_model_kd(student, teacher, train_loader, val_loader, anchors_center, anchors_corner, device,
                    epochs=STUDENT_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                    selection_metric=SELECTION_METRIC, checkpoint_name="best_model.pt",
                    temperature=KD_TEMPERATURE, alpha=KD_ALPHA, beta=KD_BETA):
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    criterion = DistillationLoss(neg_pos_ratio=NEG_POS_RATIO, temperature=temperature,
                                   alpha=alpha, beta=beta).to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"total": [], "loc": [], "cls": [], "soft_cls": [], "soft_loc": [],
        "val_accuracy": [], "val_precision": [], "val_recall": [], "val_f1_macro": [],
        "iter_total": [], "iter_loc": [], "iter_cls": [], "iter_soft_cls": [], "iter_soft_loc": []}
    best_metric_value = -1.0
    best_state_dict = None
    best_epoch = -1
    epoch_bar = tqdm(range(1, epochs + 1), desc="distilling", unit="epoch")
    for epoch in epoch_bar:
        student.train()
        running = {"total": 0.0, "loc": 0.0, "cls": 0.0, "soft_cls": 0.0, "soft_loc": 0.0}
        n_batches = 0
        batch_bar = tqdm(train_loader, desc=f"epoch {epoch}/{epochs}", leave=False)
        for imgs, boxes, labels in batch_bar:
            imgs, boxes, labels = imgs.to(device), boxes.to(device), labels.to(device)
            with torch.no_grad():
                teacher_cls, teacher_loc = teacher(imgs)
            student_cls, student_loc = student(imgs)
            cls_t, loc_t, pos_m, ign_m = match_anchors_to_targets(boxes, labels, anchors_center, anchors_corner)
            total_loss, hard_loc, hard_cls, soft_cls, soft_loc = criterion(
                student_cls, student_loc, teacher_cls, teacher_loc, cls_t, loc_t, pos_m, ign_m)
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=5.0)
            optimizer.step()
            history["iter_total"].append(total_loss.item())
            history["iter_loc"].append(hard_loc.item())
            history["iter_cls"].append(hard_cls.item())
            history["iter_soft_cls"].append(soft_cls.item())
            history["iter_soft_loc"].append(soft_loc.item())
            running["total"] += total_loss.item()
            running["loc"] += hard_loc.item()
            running["cls"] += hard_cls.item()
            running["soft_cls"] += soft_cls.item()
            running["soft_loc"] += soft_loc.item()
            n_batches += 1
            batch_bar.set_postfix(total=running["total"] / n_batches,
                                   hard_cls=running["cls"] / n_batches,
                                   soft_cls=running["soft_cls"] / n_batches)
        scheduler.step()
        for k in ("total", "loc", "cls", "soft_cls", "soft_loc"):
            history[k].append(running[k] / n_batches)
        val_metrics = evaluate(student, val_loader, anchors_center, device)
        for k in ("accuracy", "precision", "recall", "f1_macro"):
            history[f"val_{k}"].append(val_metrics[k])
        is_best = val_metrics[selection_metric] > best_metric_value
        if is_best:
            best_metric_value = val_metrics[selection_metric]
            best_state_dict = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            best_epoch = epoch
        epoch_bar.set_postfix(total=f"{history['total'][-1]:.3f}",
                               val_f1=f"{val_metrics['f1_macro']:.3f}",
                               best=f"{best_metric_value:.3f}@{best_epoch}")
        print(f"epoch {epoch:2d}/{epochs}  total={history['total'][-1]:.4f}  "
              f"hard_loc={history['loc'][-1]:.4f}  hard_cls={history['cls'][-1]:.4f}  "
              f"soft_cls={history['soft_cls'][-1]:.4f}  soft_loc={history['soft_loc'][-1]:.4f}  |  "
              f"val_acc={val_metrics['accuracy']:.3f}  val_prec={val_metrics['precision']:.3f}  "
              f"val_rec={val_metrics['recall']:.3f}  val_f1={val_metrics['f1_macro']:.3f}  "
              f"(missed {val_metrics['n_missed']}/{val_metrics['n_total']})"
              f"{'  <-- best so far' if is_best else ''}")
    print(f"\nbest checkpoint: epoch {best_epoch}  {selection_metric}={best_metric_value:.4f}")
    best_model_path = os.path.join(MODELS_DIR, checkpoint_name)
    torch.save(best_state_dict, best_model_path)
    student.load_state_dict(best_state_dict)
    print(f"loaded best checkpoint into `student_model` and saved it to {best_model_path}")
    return history, best_epoch, best_metric_value

def kd_student_job(seed, epochs=STUDENT_EPOCHS):
    global MODELS_DIR
    tag = f"kd_0p30_aug_s{seed}"
    ckpt = os.path.join(REV_DIR, f"abl_{tag}.json")
    if os.path.exists(ckpt):
        print(tag, "cached, skipping"); return json.load(open(ckpt))
    set_seed(seed)
    tl = fresh_loader_for(TRDS_ON, seed)
    tckpt = os.path.join(PROJECT_ROOT, "models_kd", "teacher_best.pt")
    assert os.path.exists(tckpt), f"missing {tckpt}"
    teacher = SSDLiteGestureDetector(width_mult=1.0).to(device)
    teacher.load_state_dict(torch.load(tckpt, map_location=device))
    student = SSDLiteGestureDetector(width_mult=0.30).to(device)
    _old = MODELS_DIR
    MODELS_DIR = os.path.join(SEEDS_DIR, f"abl_{tag}"); os.makedirs(MODELS_DIR, exist_ok=True)
    try:
        hist, best_ep, best_f1 = train_model_kd(student, teacher, tl, val_loader,
            anchors_center_d, anchors_corner_d, device, epochs=epochs,
            checkpoint_name="best_model.pt")
    finally:
        MODELS_DIR = _old
    met = evaluate(student, val_loader, anchors_center_d, device)
    student.eval(); P = []
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            P.append(predict_batch_labels(student, imgs.to(device), anchors_center_d).cpu())
    P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
    np.save(os.path.join(REV_DIR, "preds", f"abl_{tag}.npy"), P)
    rec = {"kind": "kd", "width": 0.3, "aug": "aug", "seed": seed, "teacher": tckpt,
           "T": 2.0, "alpha": 0.5, "beta": 0.5, "best_epoch": best_ep,
           "fp32": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")},
           "n_missed": met["n_missed"]}
    json.dump(rec, open(ckpt, "w"), indent=2)
    print(f"{tag} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} -> {ckpt}")
    return rec

In [ ]:
import sys
for m in [m for m in sys.modules if m == "torch._dynamo" or m.startswith("torch._dynamo.")]:
    del sys.modules[m]
import torch._dynamo
print("dynamo ok")
def _strip(fn):
    seen = 0
    while hasattr(fn, "__wrapped__") and seen < 10:
        fn = fn.__wrapped__
        seen += 1
    return torch.no_grad()(fn)
import torch.optim.optimizer as _OO
for _cls in (torch.optim.SGD, torch.optim.AdamW):
    for _m in ("add_param_group", "step", "zero_grad"):
        _raw = _strip(getattr(_cls, _m))
        if getattr(_cls, _m) is not _raw:
            setattr(_cls, _m, _raw)
print("optimizers stripped")
import json, torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
fp = json.load(open(os.path.join(REV_DIR, "split_fingerprint.json")))
assert fp["val_label_md5"] == "d210029e2814357ca28251390535186e", "split changed!"
_final = load_from_disk(DATASET_CACHE_DIR).shuffle(seed=SEED)
_nv = max(1, int(0.1 * len(_final)))
_hf_val, _hf_train = _final.select(range(_nv)), _final.select(range(_nv, len(_final)))
TRDS_ON = HaGRIDGestureDataset(_hf_train, hagrid_label_names, augment=True)
def fresh_loader_for(ds, seed, batch=BATCH_SIZE):
    return DataLoader(ds, batch_size=batch, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))
print(f"paired ready: train={len(_hf_train)} val={len(_hf_val)}")
def train_job(kind, width, seed, aug_tag, epochs=120, force_random_init=False):
    global MODELS_DIR
    tag = f"{kind}_{width:.2f}_{aug_tag}_s{seed}".replace(".", "p")
    ckpt = os.path.join(REV_DIR, f"abl_{tag}.json")
    if os.path.exists(ckpt):
        print(tag, "cached, skipping"); return json.load(open(ckpt))
    set_seed(seed)
    tl = fresh_loader_for(TRDS_ON, seed)
    m = SSDLiteGestureDetector(width_mult=width).to(device)
    if force_random_init:
        m.apply(lambda mod: mod.reset_parameters() if isinstance(mod, (nn.Conv2d, nn.Linear, nn.BatchNorm2d)) else None)
        print("forced random init (true from-scratch at width 1.0)")
    _old = MODELS_DIR
    MODELS_DIR = os.path.join(SEEDS_DIR, f"abl_{tag}"); os.makedirs(MODELS_DIR, exist_ok=True)
    try:
        hist, best_ep, best_f1 = train_model(m, tl, val_loader, anchors_center_d, anchors_corner_d,
            device, epochs=epochs, lr=LR, weight_decay=WEIGHT_DECAY, checkpoint_name="best_model.pt")
    finally:
        MODELS_DIR = _old
    met = evaluate(m, val_loader, anchors_center_d, device)
    m.eval(); P = []
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            P.append(predict_batch_labels(m, imgs.to(device), anchors_center_d).cpu())
    P = torch.cat(P).numpy(); P = np.where(P == -1, NUM_CLASSES, P)
    np.save(os.path.join(REV_DIR, "preds", f"abl_{tag}.npy"), P)
    rec = {"kind": kind, "width": width, "aug": aug_tag, "seed": seed, "best_epoch": best_ep,
           "force_random_init": force_random_init,
           "params": sum(p.numel() for p in m.parameters()),
           "fp32": {k: float(met[k]) for k in ("accuracy","precision","recall","f1_macro")},
           "n_missed": met["n_missed"]}
    json.dump(rec, open(ckpt, "w"), indent=2)
    print(f"{tag} done: best_ep={best_ep} F1={met['f1_macro']*100:.2f} -> {ckpt}")
    return rec
SEEDS_DIR = os.path.join(PROJECT_ROOT, "models_seeds"); os.makedirs(SEEDS_DIR, exist_ok=True)
for s in (0, 1, 2):
    try:
        train_job("scratch", 1.0, s, "aug", force_random_init=True)
    except Exception as e:
        print(f"1.0 s{s} FAILED: {type(e).__name__}: {e}")
print("1.0 queue done; paste this log back")

dynamo ok
optimizers stripped
paired ready: train=3906 val=433
backbone: ImageNet-pretrained MobileNetV2 (width_mult=1.0)
forced random init (true from-scratch at width 1.0)


training:   0%|          | 0/120 [00:00<?, ?epoch/s]

epoch 1/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]

epoch  1/120  total=11.4893  loc=6.5087  cls=4.9806  |  val_acc=0.000  val_prec=0.000  val_rec=0.000  val_f1=0.000  (missed 433/433)  <-- best so far


...

epoch 116/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]

epoch 116/120  total=0.8266  loc=0.3429  cls=0.4836  |  val_acc=0.665  val_prec=0.900  val_rec=0.665  val_f1=0.763  (missed 113/433)


epoch 117/120:   0%|          | 0/62 [00:00<?, ?it/s]

validating:   0%|          | 0/7 [00:00<?, ?it/s]